In [ ]:
# Cell 1: Install all packages
!pip install streamlit pyngrok pyjwt watchdog bcrypt PyPDF2 streamlit-option-menu textstat plotly python-docx -q
!pip install sentence-transformers faiss-cpu transformers>=4.40.0 accelerate bitsandbytes spacy networkx pyvis torch langdetect beautifulsoup4 -q
!python -m spacy download en_core_web_sm -q
print('✅ All packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 123.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecti

In [ ]:
# Cell 2: Mount Google Drive & setup directories
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

APP_DIR = '/content/drive/MyDrive/PolicyNav'
os.makedirs(APP_DIR, exist_ok=True)
os.makedirs(os.path.join(APP_DIR, 'documents'), exist_ok=True)
os.makedirs(os.path.join(APP_DIR, 'graphs'), exist_ok=True)
os.environ['APP_DIR'] = APP_DIR
print(f'✅ APP_DIR = {APP_DIR}')
print(f'📁 Upload your PDFs to: {APP_DIR}/documents/')

Mounted at /content/drive
✅ APP_DIR = /content/drive/MyDrive/PolicyNav
📁 Upload your PDFs to: /content/drive/MyDrive/PolicyNav/documents/


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm import tqdm
import time
import urllib3
import os
import sqlite3

# ════════════════════════════════════════════════════════════════════════════
# SETUP
# ════════════════════════════════════════════════════════════════════════════

base_dir = '/content/drive/MyDrive/PolicyNav/documents'
os.makedirs(base_dir, exist_ok=True)
db_path = '/content/drive/MyDrive/PolicyNav/policies_meta.db'

try:
    conn = sqlite3.connect(db_path, timeout=10)
    conn.execute('PRAGMA journal_mode=WAL;')
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS policies
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  title TEXT, url TEXT UNIQUE, local_path TEXT)''')
    conn.commit()
    conn.close()
    print(f"Data directory: {base_dir}")
    print(f"Database setup at: {db_path}")
except Exception as e:
    print(f"DB Error: {e}")

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Connection": "keep-alive",
}

# Domains that timeout/fail on Colab — try once with 8s, move on immediately
# URLs are NOT removed — they still get attempted, just don't waste time retrying
SLOW_DOMAINS = {
    'pmkisan.gov.in', 'www.nabard.org', 'amrut.gov.in',
    'pmsvanidhi.mohua.gov.in', 'www.mudra.org.in', 'www.standupmitra.in',
    'mdm.nic.in', 'www.aicte-india.org', 'gem.gov.in', 'www.ujala.gov.in',
    'www.nsfdc.nic.in', 'www.sipda.gov.in', 'wcd.nic.in',
    'apprenticeshipindia.org', 'pmrpy.gov.in', 'rbsk.gov.in',
    'spb.kerala.gov.in', 'www.keralawomen.gov.in', 'financedept.tn.gov.in',
    'cms.tn.gov.in', 'it.telangana.gov.in', 'www.karnataka.gov.in',
    'raitamitra.karnataka.gov.in', 'dipp.gov.in', 'www.delhi.gov.in',
    'industries.maharashtra.gov.in', 'mssds.maharashtra.gov.in',
    'www.apindustries.gov.in', 'apita.ap.gov.in', 'aphidindu.in',
    'rajit.rajasthan.gov.in', 'upagripardarshi.gov.in', 'msme.up.gov.in',
    'ic.gujarat.gov.in', 'agri.gujarat.gov.in', 'geda.gujarat.gov.in',
    'health.delhigovt.nic.in', 'socialwelfare.delhigovt.nic.in',
    'mahaevyojana.in', 'mhada.gov.in', 'wrd.maharashtra.gov.in',
    'krishi.maharashtra.gov.in', 'tshousing.telangana.gov.in',
    'agri.telangana.gov.in', 'saubhagya.gov.in',
}

# ════════════════════════════════════════════════════════════════════════════
# SECTION 1 — CENTRAL SCRAPE PAGES
# ════════════════════════════════════════════════════════════════════════════

central_scrape_pages = {
    "PMJDY":            "https://pmjdy.gov.in/",
    "PMJAY":            "https://pmjay.gov.in/",
    "PMAY_Urban":       "https://pmay-urban.gov.in/",
    "PMFBY":            "https://pmfby.gov.in/",
    "PMKISAN":          "https://pmkisan.gov.in/",
    "JalJeevanMission": "https://jaljeevanmission.gov.in/",
    "DigitalIndia":     "https://digitalindia.gov.in/",
    "SkillIndia":       "https://www.msde.gov.in/",
    "MGNREGS":          "https://nrega.nic.in/",
    "NationalHealth":   "https://nhm.gov.in/",
    "MidDayMeal":       "https://mdm.nic.in/",
    "WCD":              "https://wcd.nic.in/",
    "PMGSY":            "https://pmgsy.nic.in/",
    "AMRUT":            "https://amrut.gov.in/",
    "StartupIndia":     "https://www.startupindia.gov.in/",
    "SocialJustice":    "https://socialjustice.gov.in/",
    "Labour":           "https://labour.gov.in/",
    "MSME":             "https://msme.gov.in/",
    "MNRE":             "https://mnre.gov.in/",
    "SagarMala":        "https://sagarmala.gov.in/",
    "Education":        "https://www.education.gov.in/",
    "MEITY":            "https://www.meity.gov.in/",
    "NITI":             "https://www.niti.gov.in/",
    "DPIIT":            "https://dpiit.gov.in/",
    "ESIC":             "https://esic.gov.in/",
}

# ════════════════════════════════════════════════════════════════════════════
# SECTION 2 — CENTRAL DIRECT PDFs
# ════════════════════════════════════════════════════════════════════════════

central_direct_pdfs = [

    # ── A. AGRICULTURE & RURAL DEVELOPMENT (14) ───────────────────────
    ("PMFBY",               "https://pmfby.gov.in/pdf/Revamped%20Operational%20Guidelines_17th%20August%202020.pdf"),
    ("PMKISAN",             "https://pmkisan.gov.in/Documents/RevisedPM-KISANOperationalGuidelines(English).pdf"),
    ("SHC_Scheme",          "https://www.agriwelfare.gov.in/sites/default/files/Guidelines_for_Demonstrations_under_SHC_Scheme.pdf"),
    ("PMAY_Gramin",         "https://pmayg.dord.gov.in/netiayHome/Uploaded/Guidelines-English_Book_Final.pdf"),
    ("SwachhBharat",        "https://archive.ids.ac.uk/clts/sites/communityledtotalsanitation.org/files/SwachhBharatMissionGraminGuidelines.pdf"),
    ("JalJeevanMission",    "https://jaljeevanmission.gov.in/sites/default/files/guideline/JJM_Operational_Guidelines.pdf"),
    ("MGNREGS",             "https://nrega.nic.in/Circular_Archive/archive/MasterCircular_MGNREGA2023.pdf"),
    ("PMGSY_Guidelines",    "https://pmgsy.nic.in/Pmgsy/Upload/English/2144PMGSY_Guidelines_English.pdf"),
    ("RKVY_Guidelines",     "https://rkvy.da.gov.in/static/download/pdf/RKVY_Revised_Guidelines.pdf"),
    ("NFSM_Guidelines",     "https://nfsm.gov.in/Guidelines/NFSM_Guidelines2021.pdf"),
    ("PMKSY_Guidelines",    "https://pmksy.gov.in/microirrigation/Archive/106741614838978PMKSY_MI_Operational_Guidelines_ENG.pdf"),
    ("eNAM_Guidelines",     "https://enam.gov.in/web/docs/enam-guidelines.pdf"),
    ("ATMA_Scheme",         "https://www.manage.gov.in/pdf/ATMA_Guidelines.pdf"),
    ("KCC_Guidelines",      "https://www.nabard.org/auth/writereaddata/tender/2408202334Revised-KCC-Scheme.pdf"),

    # ── B. HOUSING & URBAN DEVELOPMENT (7) ────────────────────────────
    ("PMAY_Urban",          "https://pmay-urban.gov.in/uploads/guidelines/Operational-Guidelines-of-PMAY-U.pdf"),
    ("SmartCities",         "https://mohua.gov.in/dataSmartCities/uploads/resource/resourceDoc/Resource_Doc_1723187622_Making_a_City_Smart_Learnings_from_the_Smart_Cities_Mission.pdf"),
    ("AMRUT2",              "https://amrut.gov.in/upload/uploadfiles/files/AMRUT_2_0_Guidelines.pdf"),
    ("PMSVANidhi",          "https://pmsvanidhi.mohua.gov.in/Home/DownloadFile?FileName=PM_SVANidhi_Guidelines.pdf"),
    ("HRIDAY_Scheme",       "https://mohua.gov.in/upload/uploadfiles/files/HRIDAYSchemeGuidelines.pdf"),
    ("DAY_NULM",            "https://nulm.gov.in/PDF/NULM/SBM_Guideline_NULM.pdf"),
    ("SBM_Urban",           "https://mohua.gov.in/upload/uploadfiles/files/SBM_Urban2.0_Guidelines.pdf"),

    # ── C. FINANCIAL INCLUSION & BANKING (7) ──────────────────────────
    ("PMJDY",               "https://www.pmjdy.gov.in/files/E-Documents/PMJDY_Brochure_ENG.pdf"),
    ("PMMUDRA",             "https://www.mudra.org.in/Document/OperationalGuidelines.pdf"),
    ("StandUpIndia",        "https://www.standupmitra.in/Home/DownloadPDF?fileName=StandUpIndiaScheme.pdf"),
    ("NPS_Handbook",        "https://www.pfrda.org.in/writereaddata/links/nps%20subscriber%20handbook4175.pdf"),
    ("NFSA_Guidelines",     "https://dfpd.gov.in/writereaddata/Portal/News/580_1_NFSA-English.pdf"),
    ("PMSBY_PMJJBY",        "https://jansuraksha.gov.in/Files/PMSBY/English/Rules.pdf"),
    ("APY_Guidelines",      "https://www.pfrda.org.in/writereaddata/links/apy%20subscriber%20booklet6185.pdf"),

    # ── D. HEALTH & NUTRITION (8) ─────────────────────────────────────
    ("AyushmanBharat",      "https://pmjay.gov.in/sites/default/files/2020-09/Operational_Guidelines_PMJAY.pdf"),
    ("NHM_Framework",       "https://nhm.gov.in/images/pdf/NHM/ROP/2023-24/NHM-Framework.pdf"),
    ("PMUY_LPG",            "https://www.pmuy.gov.in/doc/pmuy_guideline.pdf"),
    ("Saubhagya",           "https://saubhagya.gov.in/Document/Saubhagya_Guidelines.pdf"),
    ("POSHAN_Abhiyan",      "https://poshanabhiyaan.gov.in/images/pdf/POSHAN-Implementation-Guideline.pdf"),
    ("Janani_Suraksha",     "https://nhm.gov.in/images/pdf/programmes/jsy/guidelines/jsy_guidelines.pdf"),
    ("RBSK_Guidelines",     "https://rbsk.gov.in/RBSKLive/PublicDocumentView/MTEz"),
    ("NCD_Programme",       "https://nhm.gov.in/images/pdf/programmes/ncd/guidelines/National-Programme-for-NCD.pdf"),

    # ── E. EDUCATION & SKILL DEVELOPMENT (8) ──────────────────────────
    ("NEP_2020",            "https://www.education.gov.in/sites/upload_files/mhrd/files/NEP_Final_English_0.pdf"),
    ("PMKVY_4",             "https://www.msde.gov.in/static/uploads/2024/02/PMKVY-4.0-Guidelines_final-copy.pdf"),
    ("SamagraShiksha",      "https://samagra.education.gov.in/docs/SS_guidelines.pdf"),
    ("RTE_Act",             "https://www.education.gov.in/sites/upload_files/mhrd/files/rte/RTE_Guideline.pdf"),
    ("MidDayMeal_PMPOSHAN", "https://mdm.nic.in/files/circulars/PM_POSHAN_Guidelines.pdf"),
    ("ScholarshipSC_ST",    "https://socialjustice.gov.in/writereaddata/UploadFile/Guidelines-SCA%20to%20SCSP.pdf"),
    ("NalandaScholarship",  "https://www.aicte-india.org/sites/default/files/Nalanda_scheme_guidelines.pdf"),
    ("NMMSS_Guidelines",    "https://www.education.gov.in/sites/upload_files/mhrd/files/nmmss-guidelines.pdf"),

    # ── F. DIGITAL & TECHNOLOGY (6) ───────────────────────────────────
    ("DigitalIndia",        "https://www.meity.gov.in/static/uploads/2024/03/Brochure.pdf"),
    ("PMGatiShakti",        "https://www.niti.gov.in/sites/default/files/2021-10/PM-GatiShakti-NMP-Report.pdf"),
    ("BharatNet",           "https://dot.gov.in/sites/default/files/BharatNet%20Phase%20II%20Guidelines.pdf"),
    ("UMANG_App",           "https://web.umang.gov.in/landing/pdf/umang-brochure.pdf"),
    ("DigiLocker",          "https://www.meity.gov.in/writereaddata/files/DigiLocker_Guidelines_2016.pdf"),
    ("NationalAI_Policy",   "https://www.meity.gov.in/writereaddata/files/NationalAIPolicyIndia.pdf"),

    # ── G. ENTERPRISE, STARTUP & INDUSTRY (8) ─────────────────────────
    ("StartupIndia",        "https://www.startupindia.gov.in/content/dam/invest-india/Templates/public/198117.pdf"),
    ("AIM_Report",          "https://aim.gov.in/pdf/AIM_Annual_Report_2022-23.pdf"),
    ("MakeInIndia",         "https://www.makeinindia.com/documents/MakeInIndia-brochure.pdf"),
    ("PLI_Scheme",          "https://dpiit.gov.in/sites/default/files/pli-scheme-guidelines-2021.pdf"),
    ("MSME_Policy",         "https://msme.gov.in/sites/default/files/MSMED_Act_amended.pdf"),
    ("GeM_Guidelines",      "https://gem.gov.in/resources/pdf/GeM-Buyer-Handbook.pdf"),
    ("NationalIP_Policy",   "https://ipindia.gov.in/writereaddata/Portal/IPOContentDocument/1_81_1_national-ip-policy-2016-book.pdf"),
    ("NationalIndustrial",  "https://dpiit.gov.in/sites/default/files/NationalIndustrialPolicy2011.pdf"),

    # ── H. ENERGY & ENVIRONMENT (7) ───────────────────────────────────
    ("NationalSolarMission","https://mnre.gov.in/img/documents/uploads/file_f-1496122248287.pdf"),
    ("KUSUM_Scheme",        "https://mnre.gov.in/img/documents/uploads/file_f-1583467099764.pdf"),
    ("PM_KUSUM_Guidelines", "https://mnre.gov.in/img/documents/uploads/1592984986905.pdf"),
    ("NationalWindPolicy",  "https://mnre.gov.in/img/documents/uploads/file_f-1560839734213.pdf"),
    ("NationalClimateNAP",  "https://moef.gov.in/wp-content/uploads/2022/08/NAPCC-report.pdf"),
    ("GreenHydrogen_Policy","https://mnre.gov.in/img/documents/uploads/file_f-1644467618034.pdf"),
    ("UJALA_Scheme",        "https://www.ujala.gov.in/download/UJALAGuidelines.pdf"),

    # ── I. SOCIAL WELFARE & INCLUSION (8) ─────────────────────────────
    ("SocialJustice_AR",    "https://socialjustice.gov.in/writereaddata/UploadFile/32691723633555.pdf"),
    ("VCF_ScheduledCastes", "https://socialjustice.gov.in/public/ckeditor/upload/56961654673488.pdf"),
    ("NSFDC_Guidelines",    "https://www.nsfdc.nic.in/pdf/scheme_guidelines.pdf"),
    ("SCA_SCSP",            "https://socialjustice.gov.in/writereaddata/UploadFile/Guidelines-SCA%20to%20SCSP.pdf"),
    ("PWD_SIPDA",           "https://www.sipda.gov.in/writereaddata/UploadFile/SIPDA_guidelines.pdf"),
    ("SeniorCitizen_IPSRC", "https://socialjustice.gov.in/writereaddata/UploadFile/34401730044145.pdf"),
    ("NMBS_Maternity",      "https://wcd.nic.in/sites/default/files/NMBS-Guidelines.pdf"),
    ("ICDS_Guidelines",     "https://wcd.nic.in/sites/default/files/ICDS_Scheme_Guidelines.pdf"),

    # ── J. LABOUR & EMPLOYMENT (5) ────────────────────────────────────
    ("LabourMinistry_MPR",  "https://labour.gov.in/sites/default/files/mpr_march_2023.pdf"),
    ("ESIC_Guidelines",     "https://esic.gov.in/attachments/publicationfile/c1455a9ac427c97e33f62b3d76ec5a50.pdf"),
    ("EPF_Handbook",        "https://epfindia.gov.in/site_docs/PDFs/Misc_PDFs/EPF_Handbook.pdf"),
    ("eShram_Guidelines",   "https://eshram.gov.in/wp-content/uploads/2021/08/eShram-Operational-Guidelines.pdf"),
    ("NAPS_Apprenticeship", "https://apprenticeshipindia.org/pdf/NAPS_Guidelines.pdf"),

    # ── K. INFRASTRUCTURE & TRANSPORT (5) ─────────────────────────────
    ("PMGSY_Rural_Roads",   "https://pmgsy.nic.in/Pmgsy/Upload/English/2144PMGSY_Guidelines_English.pdf"),
    ("UDAN_Aviation",       "https://www.aai.aero/sites/default/files/udan/UDAN_Guidelines.pdf"),
    ("SagarMala",           "https://sagarmala.gov.in/sites/default/files/SagarMala_Project_Guidelines.pdf"),
    ("PMRPY_Employment",    "https://pmrpy.gov.in/sites/default/files/pmrpy-scheme-guidelines.pdf"),
    ("NationalRailPlan",    "https://indianrailways.gov.in/railwayboard/uploads/directorate/stat_econ/National_Rail_Plan/NRP_Final_311221.pdf"),

    # ── L. NITI AAYOG REPORTS (5) ─────────────────────────────────────
    ("NITIAayog_AR_2324",   "https://www.niti.gov.in/sites/default/files/2024-05/Annual%20report%202023-24%20-%20Manual%20-14.pdf"),
    ("NITIAayog_AR_2425",   "https://www.niti.gov.in/sites/default/files/2025-02/Annual%20Report%202024-25%20English_FINAL_LOW%20RES_0.pdf"),
    ("NITIAayog_Fiscal",    "https://www.niti.gov.in/sites/default/files/2025-01/Fiscal_Health_Index_24012025_Final.pdf"),
    ("NITIAayog_ITI",       "https://www.niti.gov.in/sites/default/files/2023-02/ITI_Report_02022023_0.pdf"),
    ("NITIAayog_AR_2223",   "https://www.niti.gov.in/sites/default/files/2023-02/Annual-Report-2022-2023-English_06022023_compressed.pdf"),
]

# ════════════════════════════════════════════════════════════════════════════
# SECTION 3 — STATE GOVERNMENT PDFs
# ════════════════════════════════════════════════════════════════════════════

state_direct_pdfs = {

    "Kerala": [
        ("Kerala_IndustrialPolicy",     "https://industry.kerala.gov.in/images/pdf/2023/IND_POLICY_ENG.pdf"),
        ("Kerala_14thFiveYearPlan",     "https://spb.kerala.gov.in/sites/default/files/inline-files/14th%20Five%20Year%20Plan%2022-27%20English.pdf"),
        ("Kerala_HousingPolicy",        "https://pmay-urban.gov.in/material/component1/Kerala%20State%20Houisng%20Policy.pdf"),
        ("Kerala_AgriSchemes",          "https://www.manage.gov.in/fpoacademy/SGSchemes/kerala.pdf"),
        ("Kerala_WomenSchemes",         "https://www.keralawomen.gov.in/sites/default/files/2020-02/Schemes-for-Kerala-Women(1).pdf"),
        ("Kerala_PalliativeCarePolicy", "https://palliumindia.org/wp-content/uploads/2020/05/Kerala-State-Palliative-Care-Policy-2019.pdf"),
        ("Kerala_ITPolicy",             "https://ksitil.kerala.gov.in/wp-content/uploads/2023/09/IT-Policy-2023.pdf"),
    ],

    "TamilNadu": [
        ("TN_BudgetHighlights2024",     "https://www.tnbudget.tn.gov.in/tnweb_files/budget%20highlights/HIGHLIGHTS%20ENG%202024_25.pdf"),
        ("TN_CitizensGuideBudget",      "https://www.tnbudget.tn.gov.in/tnweb_files/CB%202024_2025_English.pdf"),
        ("TN_HealthPolicyNote",         "https://cms.tn.gov.in/sites/default/files/documents/hfw_e_pn_2023_24.pdf"),
        ("TN_HigherEducationPolicy",    "https://cms.tn.gov.in/cms_migrated/document/docfiles/hedu_e_pn_2024_25.pdf"),
        ("TN_IndustriesPolicyNote",     "https://cms.tn.gov.in/sites/default/files/documents/ind_major_e_pn_2023_24_0.pdf"),
        ("TN_DifferentlyAbledPolicy",   "https://www.scd.tn.gov.in/pdf/wda_e_pn_2023_24.pdf"),
        ("TN_SpaceIndustrialPolicy",    "https://tidco.com/wp-content/uploads/2024/06/Draft%20Tamil%20Nadu%20Space%20Industrial%20Policy.pdf"),
        ("TN_GovtSchemesVolume1",       "https://www.tnpscthervupettagam.com/wp-content/uploads/2018/06/Schemes-Volume-1-English.pdf"),
    ],

    "Telangana": [
        ("TS_MSMEPolicy2024",           "https://uncomplycate.com/wp-content/uploads/2024/10/Govt-of-Telangana-MSME-Policy-Order.pdf"),
        ("TS_IndustrialPolicy",         "https://www.telangana.gov.in/wp-content/uploads/2023/07/Telangana-Industrial-Policy-2023.pdf"),
        ("TS_ITPolicy",                 "https://it.telangana.gov.in/wp-content/uploads/2021/12/Telangana-ICT-Policy-2021-26.pdf"),
        ("TS_AgriPolicy",               "https://agri.telangana.gov.in/sites/default/files/2022-08/Agricultural_Policy_TS.pdf"),
        ("TS_WelfareSchemes",           "https://www.nirdpr.org.in/nird_docs/sagy/Telangana.pdf"),
        ("TS_EV_Policy",                "https://tsiic.telangana.gov.in/assets/pdf/ev-policy-2020.pdf"),
    ],

    "Karnataka": [
        ("KA_IndustrialPolicy2020",     "https://investkarnataka.co.in/wp-content/uploads/2020/07/Industrialpolicy.pdf"),
        ("KA_IndustrialPolicy2025",     "https://investkarnataka.co.in/wp-content/uploads/2025/02/IndustrialPolicy2025_PrintPagesSingle_.pdf"),
        ("KA_AerospaceDefencePolicy",   "https://investkarnataka.co.in/wp-content/uploads/2025/02/Karnataka-Aerospace-Defence-Policy-22-27_compressed.pdf"),
        ("KA_EV_Policy",                "https://investkarnataka.co.in/wp-content/uploads/2021/09/KarnatakaEVPolicy2021-26.pdf"),
        ("KA_ITBTPolicy",               "https://www.karnataka.gov.in/storage/pdf-files/IT-BT%20Policy%202020-2025.pdf"),
        ("KA_AgriPolicy",               "https://raitamitra.karnataka.gov.in/DocumentsNew/Agri_Policy.pdf"),
    ],

    "Delhi": [
        ("Delhi_IndustrialPolicy",      "https://dpiit.gov.in/sites/default/files/DelhiIndustrialPolicy2010-21.pdf"),
        ("Delhi_EV_Policy",             "https://ev.delhi.gov.in/pdf/ev-policy.pdf"),
        ("Delhi_SolarPolicy",           "https://www.delhi.gov.in/sites/default/files/Power/solar-policy.pdf"),
        ("Delhi_ExcisePolicy",          "https://excise.delhi.gov.in/sites/default/files/Excise/excisepolicydelhiasnotifiedon17112021.pdf"),
        ("Delhi_EducationPolicy",       "https://dsel.education.delhi.gov.in/sites/default/files/Notice/delhi_education_policy_2021.pdf"),
    ],

    "Maharashtra": [
        ("MH_IndustrialPolicy2019",     "https://industries.maharashtra.gov.in/site/upload/GR/IndustrialPolicy2019.pdf"),
        ("MH_AgriPolicy",               "https://krishi.maharashtra.gov.in/Site/Upload/GR/NewAgriPolicyEng.pdf"),
        ("MH_HousingPolicy",            "https://mhada.gov.in/sites/default/files/inline-files/HousingPolicy.pdf"),
        ("MH_WaterPolicy",              "https://wrd.maharashtra.gov.in/Site/Upload/GR/StateWaterPolicy.pdf"),
        ("MH_EV_Policy",                "https://mahadiscom.in/wp-content/uploads/2021/07/EV-Policy-2021.pdf"),
        ("MH_StartupPolicy",            "https://industries.maharashtra.gov.in/site/upload/GR/StartupMaharashtraPolicy.pdf"),
        ("MH_ITPolicy",                 "https://maharashtra.gov.in/Site/Upload/Acts%20Rules/English/IT_Policy_2023.pdf"),
    ],

    "AndhraPradesh": [
        ("AP_IndustrialPolicy",         "https://www.apindustries.gov.in/APIndus/UserInterface/IndustrialPolicies/AP_Industrial_Policy_2023-27.pdf"),
        ("AP_AgriPolicy",               "https://www.apagrisnet.gov.in/documents/AgriPolicy.pdf"),
        ("AP_ITPolicy",                 "https://apita.ap.gov.in/docs/AP-ICT-Policy-2021-24.pdf"),
        ("AP_EV_Policy",                "https://apindustries.gov.in/APIndus/UserInterface/IndustrialPolicies/AP_EV_and_Energy_Storage_Policy_2023.pdf"),
        ("AP_WelfareSchemes",           "https://www.nirdpr.org.in/nird_docs/sagy/AndhraPradesh.pdf"),
    ],

    "Rajasthan": [
        ("RJ_IndustrialPolicy",         "https://industries.rajasthan.gov.in/content/dam/industries/pdf/IndustrialPolicy2022.pdf"),
        ("RJ_AgriPolicy",               "https://agriculture.rajasthan.gov.in/content/dam/agriculture/RajasthanAgriPolicy.pdf"),
        ("RJ_SolarEnergyPolicy",        "https://energy.rajasthan.gov.in/content/dam/energy/Solar_Policy_2019.pdf"),
        ("RJ_StartupPolicy",            "https://rajasthan.gov.in/sites/default/files/2022-01/Startup_Policy_2022.pdf"),
    ],

    "UttarPradesh": [
        ("UP_IndustrialPolicy",         "https://invest.up.gov.in/wp-content/uploads/2020/07/Industrial-Investment-and-Employment-Promotion-Policy-2017-22.pdf"),
        ("UP_MSMEPolicy",               "https://msme.up.gov.in/files/msme_policy_2022.pdf"),
        ("UP_StartupPolicy",            "https://invest.up.gov.in/wp-content/uploads/UP-Startup-Policy-2020.pdf"),
        ("UP_EV_Policy",                "https://invest.up.gov.in/wp-content/uploads/EV-Manufacturing-Policy-2022.pdf"),
    ],

    "Gujarat": [
        ("GJ_IndustrialPolicy",         "https://ic.gujarat.gov.in/uploads/industrial-policy/Gujarat_Industrial_Policy_2020.pdf"),
        ("GJ_AgriPolicy",               "https://agri.gujarat.gov.in/documents/AgriPolicy2023.pdf"),
        ("GJ_ITPolicy",                 "https://dst.gujarat.gov.in/uploads/Gujarat_IT-ITeS_Policy_2022-27.pdf"),
        ("GJ_GreenEnergyPolicy",        "https://geda.gujarat.gov.in/images/pdf/GujSolarPowerPolicy.pdf"),
        ("GJ_StartupPolicy",            "https://ic.gujarat.gov.in/uploads/startup-policy/Gujarat_Startup_Policy_2022-27.pdf"),
    ],
}

# ════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════

def find_pdfs(page_url):
    domain = urlparse(page_url).netloc
    # Slow domains — skip scraping, direct PDFs still attempted separately
    if domain in SLOW_DOMAINS:
        return []
    print(f"\n  Scanning: {page_url}")
    try:
        response = requests.get(page_url, headers=HEADERS, timeout=30, verify=False)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        pdf_links = set()
        for link in soup.find_all("a", href=True):
            if ".pdf" in link["href"].lower():
                pdf_links.add(urljoin(page_url, link["href"]))
        return list(pdf_links)
    except Exception as e:
        print(f"  ⚠ Scrape failed ({domain}): {type(e).__name__}")
        return []


def is_already_downloaded(url, filepath):
    try:
        if os.path.exists(filepath):
            return True
        conn = sqlite3.connect(db_path, timeout=10)
        c = conn.cursor()
        c.execute("SELECT id FROM policies WHERE url=?", (url,))
        res = c.fetchone()
        conn.close()
        return res is not None
    except:
        return False


def record_download(save_name, url, filepath):
    try:
        conn = sqlite3.connect(db_path, timeout=10)
        c = conn.cursor()
        c.execute("INSERT OR IGNORE INTO policies (title, url, local_path) VALUES (?, ?, ?)",
                  (save_name, url, filepath))
        conn.commit()
        conn.close()
    except Exception as e:
        print(f"  DB Write Error: {e}")


def download_pdf(url, scheme_name):
    domain = urlparse(url).netloc
    filename = url.split("/")[-1].split("?")[0]
    if not filename.endswith(".pdf"):
        filename = scheme_name + ".pdf"
    save_name = f"{scheme_name}_{filename}" if not filename.startswith(scheme_name) else filename
    filepath = os.path.join(base_dir, save_name)

    if is_already_downloaded(url, filepath):
        print(f"  ⏭ Skip (exists): {save_name}")
        return

    # ── SLOW domains: 1 attempt, 8s timeout, move on ──────────────────
    if domain in SLOW_DOMAINS:
        try:
            r = requests.get(url, headers=HEADERS, stream=True, timeout=8, verify=False)
            ct = r.headers.get("Content-Type", "").lower()
            if "application/pdf" not in ct and not r.content[:4].startswith(b'%PDF'):
                print(f"  ⚠ Not a PDF: {domain}")
                return
            total = int(r.headers.get("content-length", 0))
            with open(filepath, "wb") as f, tqdm(
                desc=save_name[:50], total=total, unit="B", unit_scale=True, leave=False
            ) as bar:
                for chunk in r.iter_content(1024):
                    f.write(chunk)
                    bar.update(len(chunk))
            record_download(save_name, url, filepath)
            print(f"  ✅ {save_name}")
        except Exception:
            print(f"  ⚠ Slow/blocked (skipping): {domain}")
        return   # always move on after 1 attempt

    # ── NORMAL domains: 2 attempts, 60s timeout ────────────────────────
    for attempt in range(2):
        try:
            r = requests.get(url, headers=HEADERS, stream=True, timeout=60, verify=False)
            ct = r.headers.get("Content-Type", "").lower()
            if "application/pdf" not in ct and not r.content[:4].startswith(b'%PDF'):
                print(f"  ⚠ Not a PDF: {url}")
                return
            total = int(r.headers.get("content-length", 0))
            with open(filepath, "wb") as f, tqdm(
                desc=save_name[:50], total=total, unit="B", unit_scale=True, leave=False
            ) as bar:
                for chunk in r.iter_content(1024):
                    f.write(chunk)
                    bar.update(len(chunk))
            record_download(save_name, url, filepath)
            print(f"  ✅ {save_name}")
            return
        except Exception as e:
            if attempt == 1:
                print(f"  ❌ {scheme_name} → {type(e).__name__}")
                return
            time.sleep(2)


# ════════════════════════════════════════════════════════════════════════════
# DOWNLOAD PIPELINE
# ════════════════════════════════════════════════════════════════════════════

pdf_mapping = []

# 1. Central direct PDFs
print("📥 Queuing Central Government PDFs...")
for scheme_name, url in central_direct_pdfs:
    pdf_mapping.append((url, f"Central_{scheme_name}"))
print(f"   {len(central_direct_pdfs)} central direct PDFs queued")

# 2. State direct PDFs
print("\n📥 Queuing State Government PDFs...")
state_count = 0
for state, pdfs in state_direct_pdfs.items():
    for scheme_name, url in pdfs:
        pdf_mapping.append((url, f"{state}_{scheme_name}"))
        state_count += 1
print(f"   {state_count} state direct PDFs queued")

# 3. Scrape central landing pages (slow domains auto-skipped in find_pdfs)
print("\n🔍 Scraping Central Government landing pages...")
scraped_count = 0
existing_urls = {u for u, s in pdf_mapping}
for name, url in central_scrape_pages.items():
    pdfs = find_pdfs(url)
    new = [p for p in pdfs if p not in existing_urls]
    for pdf in new:
        pdf_mapping.append((pdf, f"Central_{name}"))
        existing_urls.add(pdf)
    if new:
        print(f"   {name}: +{len(new)} new PDFs found")
    scraped_count += len(new)
print(f"   {scraped_count} additional PDFs from scraping")

# Summary before download
slow_count  = sum(1 for u, s in pdf_mapping if urlparse(u).netloc in SLOW_DOMAINS)
normal_count = len(pdf_mapping) - slow_count
estimated_mins = round((normal_count * 10 + slow_count * 9) / 60)

print(f"""
📊 Queue Summary
   ─────────────────────────────────────────
   Central direct PDFs   : {len(central_direct_pdfs)}
   State direct PDFs     : {state_count}
   Scraped PDFs          : {scraped_count}
   ─────────────────────────────────────────
   TOTAL                 : {len(pdf_mapping)}

⚡ Speed settings
   Normal domains        : 2 attempts × 60s timeout
   Slow/blocked domains  : 1 attempt  × 8s  timeout (fail fast)
   Normal URLs           : {normal_count}
   Slow URLs             : {slow_count} (8s each = ~{round(slow_count*9/60)} min max)
   ─────────────────────────────────────────
   Estimated runtime     : ~{estimated_mins}–{estimated_mins+20} minutes
   ─────────────────────────────────────────
   Already downloaded files will be skipped instantly
""")

# 4. Download
print("🚀 Starting downloads...\n")
slow_failed = []
normal_failed = []

for pdf_url, scheme_name in pdf_mapping:
    domain = urlparse(pdf_url).netloc
    try:
        download_pdf(pdf_url, scheme_name)
        time.sleep(0.5)
    except Exception as e:
        if domain in SLOW_DOMAINS:
            slow_failed.append(pdf_url)
        else:
            normal_failed.append((pdf_url, str(e)))

# 5. Final summary
try:
    conn = sqlite3.connect(db_path, timeout=10)
    c = conn.cursor()
    c.execute("SELECT COUNT(*) FROM policies")
    count = c.fetchone()[0]
    conn.close()
    print(f"\n{'='*50}")
    print(f"✅ Done! Documents in database : {count}")
    print(f"   Slow domain failures        : {len(slow_failed)} (expected — download locally)")
    if normal_failed:
        print(f"   Unexpected failures         : {len(normal_failed)}")
        for url, err in normal_failed:
            print(f"   • {url[:70]} → {err}")
    print(f"{'='*50}")
    if slow_failed:
        print(f"\n💡 {len(slow_failed)} docs blocked on Colab. Run local_downloader.py")
        print(f"   on your machine and upload to: {base_dir}")
except Exception as e:
    print(f"Final DB Check Error: {e}")

Data directory: /content/drive/MyDrive/PolicyNav/documents
Database setup at: /content/drive/MyDrive/PolicyNav/policies_meta.db
📥 Queuing Central Government PDFs...
   88 central direct PDFs queued

📥 Queuing State Government PDFs...
   57 state direct PDFs queued

🔍 Scraping Central Government landing pages...

  Scanning: https://pmjdy.gov.in/
   PMJDY: +7 new PDFs found

  Scanning: https://pmjay.gov.in/

  Scanning: https://pmay-urban.gov.in/
   PMAY_Urban: +37 new PDFs found

  Scanning: https://pmfby.gov.in/

  Scanning: https://jaljeevanmission.gov.in/
   JalJeevanMission: +35 new PDFs found

  Scanning: https://digitalindia.gov.in/
   DigitalIndia: +4 new PDFs found

  Scanning: https://www.msde.gov.in/

  Scanning: https://nrega.nic.in/
  ⚠ Scrape failed (nrega.nic.in): ConnectionError

  Scanning: https://nhm.gov.in/
   NationalHealth: +16 new PDFs found

  Scanning: https://pmgsy.nic.in/
   PMGSY: +155 new PDFs found

  Scanning: https://www.startupindia.gov.in/
   StartupIn

  ✅ Central_PMFBY_Revamped%20Operational%20Guidelines_17th%20August%202020.pdf
  ⚠ Slow/blocked (skipping): pmkisan.gov.in
  ⚠ Not a PDF: https://www.agriwelfare.gov.in/sites/default/files/Guidelines_for_Demonstrations_under_SHC_Scheme.pdf
  ⚠ Not a PDF: https://pmayg.dord.gov.in/netiayHome/Uploaded/Guidelines-English_Book_Final.pdf
  ⚠ Not a PDF: https://archive.ids.ac.uk/clts/sites/communityledtotalsanitation.org/files/SwachhBharatMissionGraminGuidelines.pdf


  ✅ Central_JalJeevanMission_JJM_Operational_Guidelines.pdf
  ⚠ Not a PDF: https://nrega.nic.in/Circular_Archive/archive/MasterCircular_MGNREGA2023.pdf
  ⚠ Not a PDF: https://pmgsy.nic.in/Pmgsy/Upload/English/2144PMGSY_Guidelines_English.pdf
  ⚠ Not a PDF: https://rkvy.da.gov.in/static/download/pdf/RKVY_Revised_Guidelines.pdf
  ⚠ Not a PDF: https://nfsm.gov.in/Guidelines/NFSM_Guidelines2021.pdf
  ⚠ Not a PDF: https://pmksy.gov.in/microirrigation/Archive/106741614838978PMKSY_MI_Operational_Guidelines_ENG.pdf
  ⚠ Not a PDF: https://enam.gov.in/web/docs/enam-guidelines.pdf
  ⚠ Not a PDF: https://www.manage.gov.in/pdf/ATMA_Guidelines.pdf
  ⚠ Slow/blocked (skipping): www.nabard.org


  ✅ Central_PMAY_Urban_Operational-Guidelines-of-PMAY-U.pdf


  ✅ Central_SmartCities_Resource_Doc_1723187622_Making_a_City_Smart_Learnings_from_the_Smart_Cities_Mission.pdf
  ⚠ Slow/blocked (skipping): amrut.gov.in
  ⚠ Slow/blocked (skipping): pmsvanidhi.mohua.gov.in
  ⚠ Not a PDF: https://mohua.gov.in/upload/uploadfiles/files/HRIDAYSchemeGuidelines.pdf
  ⚠ Not a PDF: https://nulm.gov.in/PDF/NULM/SBM_Guideline_NULM.pdf
  ⚠ Not a PDF: https://mohua.gov.in/upload/uploadfiles/files/SBM_Urban2.0_Guidelines.pdf


...

  ✅ Central_ESIC_Details_of_candidates_of_Combined_Medical_Services_Examination_2024_on_the_PRATIBHA_Setu_portal_of_UPSC_English_1768300965.pdf


  ✅ Central_ESIC_Advertisement_for_Recruitment_of_Insurance_Medical_Officers_Grade_II_IMO_Gr_II_in_ESI_Corporation_English_1768300965.pdf


  ✅ Central_ESIC_448c39d79029731de55b33a230f4a832.pdf
  ❌ Central_ESIC → ConnectTimeout


  ✅ Central_ESIC_485c7209dc413fcd841d4f59191e05a4.pdf


  ✅ Central_ESIC_esic at a glance.pdf

✅ Done! Documents in database : 325
   Slow domain failures        : 0 (expected — download locally)


## Cell 3: Write `readability_utils.py`

In [ ]:
%%writefile readability_utils.py
# readability_utils.py
# readability_utils.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = max(1, text.count("."))
        self.num_words = len(text.split())
        self.char_count = len(text)
        self.syllables = textstat.syllable_count(text)
        self.complex_words = textstat.lexicon_count(text, removepunct=True)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }

## Cell 4: Write `db.py`

In [ ]:
%%writefile db.py
import sqlite3, bcrypt, datetime, os

_APP_DIR = os.environ.get('APP_DIR', '/content/drive/MyDrive/PolicyNav')
DB_NAME = os.path.join(_APP_DIR, "users.db")

def _get_conn():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

def init_db():
    conn = _get_conn(); c = conn.cursor()
    c.execute("CREATE TABLE IF NOT EXISTS users (email TEXT PRIMARY KEY, password BLOB, created_at TEXT)")
    c.execute("CREATE TABLE IF NOT EXISTS activity_history (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, activity_type TEXT, input_text TEXT, output_text TEXT, timestamp TEXT)")
    c.execute("CREATE TABLE IF NOT EXISTS feedback (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, section TEXT, rating INTEGER, comments TEXT, timestamp TEXT)")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS qa_history (
            id        INTEGER PRIMARY KEY AUTOINCREMENT,
            email     TEXT,
            question  TEXT,
            answer    TEXT,
            sources   TEXT,
            language  TEXT,
            elapsed   REAL,
            timestamp TEXT DEFAULT (datetime('now'))
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS qa_feedback (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            qa_id      INTEGER,
            email      TEXT,
            feedback   TEXT,
            timestamp  TEXT DEFAULT (datetime('now'))
        )
    """)
    conn.commit(); conn.close()

def log_activity(email, activity_type, input_text, output_text):
    conn = _get_conn(); c = conn.cursor()
    c.execute("INSERT INTO activity_history (email, activity_type, input_text, output_text, timestamp) VALUES (?, ?, ?, ?, ?)",
              (email, activity_type, input_text, output_text, datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))
    conn.commit(); conn.close()

def get_user_activity(email):
    conn = _get_conn(); c = conn.cursor()
    c.execute("SELECT activity_type, input_text, output_text, timestamp FROM activity_history WHERE email = ? ORDER BY timestamp DESC LIMIT 50", (email,))
    data = c.fetchall(); conn.close(); return data

def submit_feedback(email, section, rating, comments=""):
    conn = _get_conn(); c = conn.cursor()
    c.execute("INSERT INTO feedback (email, section, rating, comments, timestamp) VALUES (?, ?, ?, ?, ?)",
              (email, section, rating, comments, datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))
    conn.commit(); conn.close()

def register_user(email, password):
    conn = _get_conn(); c = conn.cursor()
    try:
        c.execute("INSERT INTO users (email, password, created_at) VALUES (?, ?, ?)",
                  (email, bcrypt.hashpw(password.encode(), bcrypt.gensalt()), datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))
        conn.commit(); return True
    except: return False
    finally: conn.close()

def authenticate_user(email, password):
    conn = _get_conn(); c = conn.cursor()
    c.execute("SELECT password FROM users WHERE email = ?", (email,))
    data = c.fetchone(); conn.close()
    return True if data and bcrypt.checkpw(password.encode(), data[0]) else False



Writing db.py


## Cell 5: Write `vector_store.py`

In [ ]:
%%writefile vector_store.py
import os, glob, pickle, faiss, PyPDF2
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer

def _get_paths():
    _APP_DIR = os.environ.get('APP_DIR', '.')
    return os.path.join(_APP_DIR, 'faiss_index.bin'), os.path.join(_APP_DIR, 'faiss_meta.pkl')

INDEX_PATH, META_PATH = None, None  # resolved lazily

# We only load it once cached in Streamlit
_embedder = None

def get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    return _embedder

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        for page in PyPDF2.PdfReader(pdf_path).pages: text += page.extract_text() + "\n"
    except: pass
    return text

def ingest_documents(docs_dir):
    INDEX_PATH, META_PATH = _get_paths()
    if not os.path.exists(docs_dir): return 0

    # Load existing metdata to avoid re-ingesting
    existing_metadata = []
    if os.path.exists(META_PATH):
        try:
            with open(META_PATH, 'rb') as f: existing_metadata = pickle.load(f)
        except: pass

    existing_filenames = set([d['filename'] for d in existing_metadata])

    files = glob.glob(os.path.join(docs_dir, "*"))
    new_chunks = []
    new_metadata = []

    for filepath in files:
        filename = os.path.basename(filepath)
        if filename in existing_filenames: continue # Skip already ingested!

        print(f"Parsing new document: {filename}")
        text = ""
        if filepath.lower().endswith(".pdf"): text = extract_text_from_pdf(filepath)
        elif filepath.lower().endswith((".htm", ".html")): text = BeautifulSoup(open(filepath, 'r').read(), 'html.parser').get_text(separator=' ')
        elif filepath.lower().endswith(".txt"): text = open(filepath, 'r').read()

        if text.strip():
            for i in range(0, len(text), 1500):
                chunk = text[i:i+1500]
                if len(chunk) > 50:
                    new_chunks.append(chunk)
                    new_metadata.append({"filename": filename, "content": chunk})

    if not new_chunks: return 0 # Nothing new to ingest

    embedder = get_embedder()
    embeddings = embedder.encode(new_chunks, convert_to_numpy=True)

    if os.path.exists(INDEX_PATH):
        index = faiss.read_index(INDEX_PATH)
    else:
        index = faiss.IndexFlatL2(embeddings.shape[1])

    index.add(embeddings)
    faiss.write_index(index, INDEX_PATH)

    final_metadata = existing_metadata + new_metadata
    with open(META_PATH, 'wb') as f: pickle.dump(final_metadata, f)
    return len(new_chunks)

def search_documents(query, top_k=5):
    INDEX_PATH, META_PATH = _get_paths()
    if not os.path.exists(INDEX_PATH) or not os.path.exists(META_PATH): return []
    try:
        embedder = get_embedder()
        index = faiss.read_index(INDEX_PATH)
        with open(META_PATH, 'rb') as f: metadata = pickle.load(f)
        distances, indices = index.search(embedder.encode([query], convert_to_numpy=True), top_k)

        # Deduplicate to try to get broader file sources rather than just 5 chunks from the exact same page
        results = []
        seen_files = set()
        for idx in indices[0]:
            if idx != -1 and idx < len(metadata):
                doc = metadata[idx]
                fname = doc['filename']
                if list(seen_files).count(fname) < 2: # Max 2 chunks from same file allowed
                    results.append(doc)
                    seen_files.add(fname)
        return results
    except: return []

def get_all_documents():
    INDEX_PATH, META_PATH = _get_paths()
    if not os.path.exists(META_PATH): return []
    with open(META_PATH, 'rb') as f: return pickle.load(f)



Writing vector_store.py


## Cell 6: Write `knowledge_graph.py`

In [ ]:
%%writefile knowledge_graph.py
"""
knowledge_graph.py  –  PolicyNav M3
Rich entity + relationship extraction with policy-domain colour coding,
community detection, relationship-type edges and interactive legend.
"""
import os, re, collections
import spacy, networkx as nx
from pyvis.network import Network

# ── spaCy model ────────────────────────────────────────────────────────────
try:
    nlp = spacy.load("en_core_web_lg")
except Exception:
    try:
        nlp = spacy.load("en_core_web_md")
    except Exception:
        try:
            nlp = spacy.load("en_core_web_sm")
        except Exception:
            nlp = None

# ── Entity label → display colour ──────────────────────────────────────────
ENTITY_COLORS = {
    "ORG":      "#38bdf8",   # sky-blue   – organisations
    "LAW":      "#f59e0b",   # amber      – laws / acts
    "GPE":      "#22d3ee",   # cyan       – places
    "PERSON":   "#a78bfa",   # violet     – persons
    "MONEY":    "#34d399",   # emerald    – monetary amounts
    "PERCENT":  "#34d399",
    "DATE":     "#94a3b8",   # slate      – dates
    "CARDINAL": "#94a3b8",
    "NORP":     "#f472b6",   # pink       – nationalities / groups
    "PRODUCT":  "#fb923c",   # orange     – products / schemes
    "EVENT":    "#fb923c",
    "WORK_OF_ART": "#e879f9",
    "FACILITY": "#60a5fa",
}
ENTITY_SIZES = {
    "LAW": 28, "ORG": 24, "GPE": 20, "PERSON": 20,
    "NORP": 18, "PRODUCT": 18, "EVENT": 18,
}
DEFAULT_COLOR = "#64748b"
DEFAULT_SIZE  = 14

# ── Relationship patterns ───────────────────────────────────────────────────
REL_PATTERNS = [
    (r'\b(implements?|implemented by)\b',          "implements",   "#38bdf8"),
    (r'\b(funds?|funded by|finances?|financed by)\b', "funds",    "#34d399"),
    (r'\b(regulates?|regulated by|oversees?)\b',   "regulates",   "#f59e0b"),
    (r'\b(covers?|covered under|applicable to)\b', "covers",      "#a78bfa"),
    (r'\b(benefits?|beneficiaries?)\b',            "benefits",    "#22d3ee"),
    (r'\b(established|constituted|formed)\b',      "established", "#f472b6"),
    (r'\b(amends?|amended by)\b',                  "amends",      "#fb923c"),
    (r'\b(under|pursuant to|in accordance with)\b',"under",       "#94a3b8"),
]

# ── Policy domain colour for document nodes ────────────────────────────────
DOMAIN_KEYWORDS = {
    "Agriculture":   ["agri", "farm", "crop", "kisan", "rural", "nfsm", "pmfby", "rkvy"],
    "Health":        ["health", "medical", "hospital", "nhm", "pmjay", "ayushman", "poshan"],
    "Education":     ["education", "school", "skill", "nep", "pmkvy", "samagra", "rte"],
    "Housing":       ["housing", "pmay", "urban", "smart city", "amrut", "swachh"],
    "Finance":       ["mudra", "pmjdy", "jan dhan", "msme", "startup", "loan", "bank"],
    "Energy":        ["solar", "energy", "mnre", "kusum", "wind", "green hydrogen"],
    "Labour":        ["labour", "epf", "esic", "eshram", "employment", "apprentice"],
    "Digital":       ["digital", "meity", "bharat net", "umang", "digilocker", "ai policy"],
    "Social":        ["social", "welfare", "sc ", "st ", "obc", "disability", "women"],
}
DOMAIN_COLORS = {
    "Agriculture": "#86efac", "Health": "#f9a8d4", "Education": "#fde68a",
    "Housing":     "#93c5fd", "Finance": "#6ee7b7", "Energy":    "#fcd34d",
    "Labour":      "#c4b5fd", "Digital": "#67e8f9", "Social":    "#fca5a5",
    "General":     "#475569",
}

def _doc_domain(filename: str) -> str:
    name = filename.lower()
    for domain, kws in DOMAIN_KEYWORDS.items():
        if any(k in name for k in kws):
            return domain
    return "General"

def _extract_relationships(sent_text: str, entities: list) -> list:
    """Return list of (ent_a, rel_label, ent_b, color) from a sentence."""
    rels = []
    if len(entities) < 2:
        return rels
    for pattern, label, color in REL_PATTERNS:
        if re.search(pattern, sent_text, re.IGNORECASE):
            # connect first two entities that appear in this sentence
            rels.append((entities[0], label, entities[1], color))
            break
    return rels

def build_graph_from_documents(docs, max_docs=60, entity_types=None):
    """
    Build a NetworkX graph with:
    - Document nodes (policy-domain coloured)
    - Entity nodes (type-coloured, sized by importance)
    - Typed relationship edges
    - Community detection via label propagation
    """
    if not nlp:
        return nx.Graph(), {}, {}

    if entity_types is None:
        entity_types = list(ENTITY_COLORS.keys())

    G = nx.Graph()
    entity_freq  = collections.Counter()   # global frequency
    rel_edges    = []                      # (a, label, b, color)

    docs_to_process = docs[:max_docs]

    for doc_obj in docs_to_process:
        filename = doc_obj["filename"]
        domain   = _doc_domain(filename)
        doc_node = f"📄 {filename}"

        G.add_node(
            doc_node,
            title    = f"<b>{filename}</b><br>Domain: {domain}",
            color    = DOMAIN_COLORS.get(domain, DOMAIN_COLORS["General"]),
            size     = 32,
            shape    = "star",
            group    = "document",
            domain   = domain,
        )

        # Process up to 2000 chars per doc for speed
        text  = doc_obj["content"][:2000]
        spacy_doc = nlp(text)

        # Per-sentence entity extraction + relationship detection
        for sent in spacy_doc.sents:
            sent_ents = [
                ent.text.strip()
                for ent in sent.ents
                if ent.label_ in entity_types
                and len(ent.text.strip()) > 2
                and not ent.text.strip().isdigit()
            ]
            sent_ents = list(dict.fromkeys(sent_ents))  # deduplicate, keep order

            for ent_text in sent_ents:
                entity_freq[ent_text] += 1
                ent_label = next(
                    (e.label_ for e in sent.ents if e.text.strip() == ent_text), ""
                )
                G.add_node(
                    ent_text,
                    group = "entity",
                    color = ENTITY_COLORS.get(ent_label, DEFAULT_COLOR),
                    size  = ENTITY_SIZES.get(ent_label, DEFAULT_SIZE),
                    title = f"<b>{ent_text}</b><br>Type: {ent_label}<br>Mentions: {entity_freq[ent_text] + 1}",
                    label = ent_text,
                )
                # doc → entity edge
                if not G.has_edge(doc_node, ent_text):
                    G.add_edge(doc_node, ent_text, color="#334155", width=1)

            # Extract inter-entity relationships
            for (a, label, b, color) in _extract_relationships(sent.text, sent_ents):
                if a in G and b in G:
                    rel_edges.append((a, label, b, color))

    # Add relationship edges (higher weight / different colour)
    for (a, label, b, color) in rel_edges:
        if G.has_node(a) and G.has_node(b):
            G.add_edge(a, b, title=label, color=color, width=2.5, label=label)

    # Scale node size by frequency
    for node, freq in entity_freq.items():
        if node in G.nodes and G.nodes[node].get("group") == "entity":
            base = G.nodes[node].get("size", DEFAULT_SIZE)
            G.nodes[node]["size"] = min(base + freq * 1.5, 48)

    # Community detection (label propagation, NetworkX built-in)
    try:
        communities = nx.community.label_propagation_communities(G)
        community_palette = [
            "#38bdf8","#6366f1","#22d3ee","#f59e0b",
            "#34d399","#a78bfa","#f472b6","#fb923c",
        ]
        for i, community in enumerate(communities):
            color = community_palette[i % len(community_palette)]
            for node in community:
                if G.nodes[node].get("group") == "entity":
                    G.nodes[node]["borderWidth"] = 2
                    G.nodes[node]["borderWidthSelected"] = 4
    except Exception:
        pass

    return G, entity_freq, dict(collections.Counter(
        G.nodes[n].get("group", "?") for n in G.nodes
    ))


def generate_interactive_graph(docs, entity_types=None, max_docs=60):
    """Generate and save an interactive PyVis knowledge graph HTML."""
    if not docs:
        return None, {}, {}

    G, entity_freq, stats = build_graph_from_documents(
        docs, max_docs=max_docs, entity_types=entity_types
    )

    if len(G.nodes) == 0:
        return None, {}, {}

    net = Network(
        height="650px",
        width="100%",
        bgcolor="#0f172a",
        font_color="white",
        directed=False,
    )
    net.from_nx(G)

    # Physics: Barnes-Hut for readability
    net.set_options("""
    {
      "physics": {
        "enabled": true,
        "barnesHut": {
          "gravitationalConstant": -8000,
          "centralGravity": 0.3,
          "springLength": 120,
          "springConstant": 0.04,
          "damping": 0.09
        },
        "maxVelocity": 50,
        "minVelocity": 0.1,
        "solver": "barnesHut"
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 150,
        "navigationButtons": true,
        "keyboard": true
      },
      "edges": {
        "smooth": { "type": "dynamic" },
        "font": { "size": 10, "color": "#94a3b8", "align": "middle" }
      },
      "nodes": {
        "font": { "size": 13, "face": "Inter, Arial" }
      }
    }
    """)

    APP_DIR = os.environ.get("APP_DIR", ".")
    os.makedirs(os.path.join(APP_DIR, "graphs"), exist_ok=True)
    output_path = os.path.join(APP_DIR, "graphs", "policy_kg.html")
    net.save_graph(output_path)

    return output_path, entity_freq, stats


Writing knowledge_graph.py


## Cell 7: Write `nlp_engine.py`

In [ ]:
%%writefile nlp_engine.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import vector_store

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRANSLATOR_ID = "facebook/nllb-200-distilled-600M"

model = None
tokenizer = None
translator_model = None
translator_tokenizer = None

def init_model():
    global model, tokenizer, translator_model, translator_tokenizer
    if model is None:
        print(f"Loading {MODEL_ID} in 4-bit Quantization via BitsAndBytes...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", quantization_config=quantization_config)

        print(f"Loading {TRANSLATOR_ID} for extreme speed translation...")
        translator_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR_ID)
        translator_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATOR_ID, device_map="auto")
        print("Models Loaded successfully!")

LANG_CODES = {
    "English": "eng_Latn", "Hindi": "hin_Deva", "Tamil": "tam_Taml",
    "Kannada": "kan_Knda", "Telugu": "tel_Telu", "Marathi": "mar_Deva",
    "Bengali": "ben_Beng", "Malayalam": "mal_Mlym"
}

def translate_fast(text, source_lang, target_lang):
    if source_lang == target_lang: return text
    if translator_model is None: init_model()

    src_code = LANG_CODES.get(source_lang, "eng_Latn")
    tgt_code = LANG_CODES.get(target_lang, "eng_Latn")

    translator_tokenizer.src_lang = src_code
    inputs = translator_tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(translator_model.device)

    tgt_token_id = translator_tokenizer.convert_tokens_to_ids(tgt_code)
    if tgt_token_id == translator_tokenizer.unk_token_id or tgt_token_id == 0:
        print(f"[translate_fast] Warning: unknown token for {target_lang}, returning as-is.")
        return text
    outputs = translator_model.generate(**inputs, forced_bos_token_id=tgt_token_id, max_length=512)
    return translator_tokenizer.decode(outputs[0], skip_special_tokens=True)

def generate_english_response(prompt_text):
    if model is None: init_model()
    messages = [
        {
            "role": "system",
            "content": (
                "You are PolicyNav, an expert AI assistant for Indian government schemes and public policy.\n"
                "Rules:\n"
                "1. Answer ONLY from the context provided. Never use outside knowledge.\n"
                "2. Always cite the document name you got the information from.\n"
                "3. If the context lacks enough information, say: 'The provided documents do not cover this. Please refer to [scheme name] official website.'\n"
                "4. Keep answers structured: one clear paragraph or 2-3 bullet points.\n"
                "5. Never hallucinate eligibility criteria, amounts, or dates not present in context."
            )
        },
        {"role": "user", "content": prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=400,
            do_sample=True,
            temperature=0.2,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def answer_policy_question(native_question, target_lang="English", simplify=False):
    english_query = translate_fast(native_question, target_lang, "English")

    docs = vector_store.search_documents(english_query, top_k=8)
    if not docs:
        docs_context = "No relevant policies found in the knowledge base."
    else:
        context_parts = []
        for d in docs:
            state_tag = f" [{d.get('state', '').title()} state]" if d.get('state') else ""
            context_parts.append(f"--- From: {d['filename']}{state_tag} ---\n{d['content']}")
        docs_context = "\n\n".join(context_parts)

    simplify_instruction = (
        "\nIMPORTANT: Use very simple language. Avoid jargon. "
        "Explain as if talking to someone with no policy background."
    ) if simplify else ""

    prompt = (
        f"CONTEXT:\n{docs_context}\n\n"
        f"QUESTION: {english_query}\n\n"
        f"Answer the question using ONLY the context above. "
        f"Start your answer by naming the relevant scheme/document. "
        f"Be specific about amounts, dates, and eligibility if present in context."
        f"{simplify_instruction}"
    )

    eng_ans = generate_english_response(prompt)
    final_ans = translate_fast(eng_ans, "English", target_lang)
    return final_ans, docs

def summarize_document(text, target_lang="English"):
    eng_sum = generate_english_response(f"Summarize this policy into 3 highly concise bullet points:\n\n{text[:3000]}")
    return translate_fast(eng_sum, "English", target_lang)



Writing nlp_engine.py


## Cell 8: Write `app.py` (PolicyNav UI)

In [ ]:
# --- Write Full Integrated App ---
%%writefile app.py
import streamlit as st
import collections
import jwt
import datetime
import re
import sqlite3
import os
import io
import random
import smtplib
from email.message import EmailMessage
import time
import bcrypt
import PyPDF2
from streamlit_option_menu import option_menu
import plotly.graph_objects as go
from readability_utils import ReadabilityAnalyzer
try:
    import nlp_engine
    import vector_store
    import knowledge_graph
    NLP_OK = True
except ImportError:
    NLP_OK = False
TRANSFORMERS_OK = False  # Summarization handled by nlp_engine (Qwen+NLLB)


JWT_SECRET_KEY = os.getenv("JWT_SECRET_KEY")
EMAIL_ID = os.getenv("EMAIL_ID")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")
ADMIN_EMAIL_ID = os.getenv("ADMIN_EMAIL_ID")
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD")

if not JWT_SECRET_KEY:
    raise RuntimeError("JWT_SECRET_KEY not set")

if "pending_signup" not in st.session_state:
    st.session_state["pending_signup"] = None

if not EMAIL_ID or not EMAIL_APP_PASSWORD:
    raise RuntimeError("Email credentials not set. OTP email cannot be sent.")

if 'page' not in st.session_state:
    st.session_state['page'] = 'login'

# --- Configuration ---
SECRET_KEY = JWT_SECRET_KEY
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# --- SQLite Configuration ---
_APP_DIR = os.environ.get('APP_DIR', '/content/drive/MyDrive/PolicyNav')
POLICYNAV_DB = os.path.join(_APP_DIR, "policynav.db")

def get_db_connection():
    conn = sqlite3.connect(POLICYNAV_DB, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            email TEXT UNIQUE NOT NULL,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer TEXT NOT NULL,
            role TEXT DEFAULT 'user',
            created_at TEXT DEFAULT CURRENT_TIMESTAMP,
            last_login TEXT DEFAULT NULL,
            failed_attempts INTEGER DEFAULT 0,
            lock_until REAL DEFAULT NULL
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS password_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            password TEXT NOT NULL,
            changed_at REAL NOT NULL,
            FOREIGN KEY (user_id) REFERENCES users(id)
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS otp_codes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            email TEXT NOT NULL,
            otp_hash TEXT NOT NULL,
            expires_at REAL NOT NULL,
            attempts INTEGER DEFAULT 0,
            created_at REAL NOT NULL
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS qa_history (
            id        INTEGER PRIMARY KEY AUTOINCREMENT,
            email     TEXT NOT NULL,
            question  TEXT NOT NULL,
            answer    TEXT NOT NULL,
            sources   TEXT,
            language  TEXT DEFAULT 'English',
            elapsed   REAL,
            timestamp TEXT DEFAULT CURRENT_TIMESTAMP
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS qa_feedback (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            history_id INTEGER,
            email      TEXT NOT NULL,
            question   TEXT NOT NULL,
            rating     INTEGER NOT NULL,
            comment    TEXT DEFAULT '',
            timestamp  TEXT DEFAULT CURRENT_TIMESTAMP
        )
    ''')

    conn.commit()
    conn.close()

init_db()


## --- AUTH HELPERS ---
# --- Password Hashing Utils ---
def hash_password(password: str) -> str:
    return bcrypt.hashpw(
        password.encode(),
        bcrypt.gensalt(rounds=12)  # explicit cost factor
    ).decode()

def ensure_admin_exists():
    if not ADMIN_EMAIL_ID or not ADMIN_PASSWORD:
        return

    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM users WHERE email = ?", (ADMIN_EMAIL_ID,))
    admin = cursor.fetchone()

    if not admin:
        hashed_password = hash_password(ADMIN_PASSWORD)

        cursor.execute('''
            INSERT INTO users (email, username, password,
                               security_question, security_answer, role)
            VALUES (?, ?, ?, ?, ?, 'admin')
        ''', (
            ADMIN_EMAIL_ID,
            "admin",
            hashed_password,
            "System Generated",
            "admin"
        ))

        conn.commit()

    conn.close()

ensure_admin_exists()

# --- OTP Management (Milestone 2) ---
def otp_verify_page():

    if "otp_email" not in st.session_state:
      st.error("OTP session expired. Please login/signup again.")
      st.session_state["page"] = "login"
      st.rerun()

    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([2, 3, 2])

    with col2:
        st.markdown(
            "<h2 style='text-align:center;color:#38bdf8;'>OTP Verification</h2>",
            unsafe_allow_html=True
        )

        st.markdown(
            "<p style='text-align:center;color:#94a3b8;'>"
            "Enter the 6-digit OTP sent to your email</p>",
            unsafe_allow_html=True
        )

        record = get_otp(st.session_state.get("otp_email"))

        if not record:
            st.error("OTP not found. Please request a new OTP.")
            st.stop()

        # Block after too many attempts
        if record["attempts"] >= 5:
            delete_otp(st.session_state["otp_email"])

            st.error("Too many incorrect attempts. OTP blocked. Redirecting to login...")

            # Clean OTP-related session state
            st.session_state.pop("otp_email", None)
            st.session_state.pop("otp_context", None)

            time.sleep(2)
            st.session_state["page"] = "login"
            st.rerun()

        with st.form("otp_verify_form"):
            entered_otp = st.text_input(
                "OTP",
                max_chars=6,
                placeholder="••••••"
            ).strip()

            verify = st.form_submit_button("Verify OTP")

        if verify:
            # Expiry check
            if time.time() > record["expires_at"]:
                delete_otp(st.session_state["otp_email"])
                st.error("OTP expired. Please request a new OTP.")
                st.stop()

            # Invalid OTP
            if not entered_otp.isdigit() or len(entered_otp) != 6:
                st.error("OTP must be a 6-digit number.")
                st.stop()

            if not bcrypt.checkpw(
                entered_otp.encode(),
                record["otp_hash"].encode()
            ):
                conn = get_db_connection()
                cursor = conn.cursor()
                cursor.execute(
                    "UPDATE otp_codes SET attempts = attempts + 1 WHERE email = ?",
                    (st.session_state["otp_email"],)
                )
                conn.commit()
                conn.close()

                st.error("Invalid OTP")
                st.stop()

            # OTP VERIFIED SUCCESSFULLY
            delete_otp(st.session_state["otp_email"])

            if st.session_state.get("otp_context") == "forgot":
                reset_token = create_access_token({
                    "email": st.session_state["otp_email"],
                    "purpose": "password_reset"
                })

                st.session_state["reset_token"] = reset_token
                st.session_state.pop("otp_email", None)
                st.session_state.pop("otp_context", None)

                st.session_state["page"] = "reset"
                st.rerun()


def generate_otp():
    return str(random.randint(100000, 999999))

def send_otp(email, otp):
    msg = EmailMessage()

    # ---- SUBJECT ----
    msg["Subject"] = "🔐 PolicyNav OTP Verification"
    msg["From"] = EMAIL_ID
    msg["To"] = email

    # ---- HTML EMAIL BODY ----
    html_content = f'''
    <html>
    <body style="
        margin:0;
        padding:0;
        background-color:#020617;
        font-family: 'Inter', 'Segoe UI', Arial, sans-serif;
    ">
        <table width="100%" cellpadding="0" cellspacing="0">
            <tr>
                <td align="center" style="padding:40px 0;">
                    <table width="520" cellpadding="0" cellspacing="0" style="
                        background:#0f172a;
                        border-radius:16px;
                        padding:32px;
                        box-shadow:0 20px 40px rgba(0,0,0,0.6);
                    ">

                        <!-- HEADER -->
                        <tr>
                            <td align="center" style="padding-bottom:12px;">
                                <h1 style="
                                    margin:0;
                                    font-size:28px;
                                    font-weight:800;
                                    background:linear-gradient(90deg,#38bdf8,#6366f1,#22d3ee);
                                    -webkit-background-clip:text;
                                    -webkit-text-fill-color:transparent;
                                ">
                                    PolicyNav
                                </h1>
                            </td>
                        </tr>

                        <tr>
                            <td align="center" style="padding-bottom:24px;">
                                <p style="
                                    margin:0;
                                    color:#94a3b8;
                                    font-size:15px;
                                ">
                                    Secure OTP Verification
                                </p>
                            </td>
                        </tr>

                        <!-- OTP BOX -->
                        <tr>
                            <td align="center" style="padding:24px 0;">
                                <div style="
                                    display:inline-block;
                                    padding:18px 36px;
                                    font-size:32px;
                                    letter-spacing:6px;
                                    font-weight:700;
                                    color:#ffffff;
                                    background:linear-gradient(135deg,#38bdf8,#6366f1);
                                    border-radius:12px;
                                ">
                                    {otp}
                                </div>
                            </td>
                        </tr>

                        <!-- INFO -->
                        <tr>
                            <td align="center" style="padding-top:12px;">
                                <p style="
                                    margin:0;
                                    color:#e5e7eb;
                                    font-size:14px;
                                ">
                                    This OTP is valid for <b>2 minutes</b>.
                                </p>
                            </td>
                        </tr>

                        <tr>
                            <td align="center" style="padding-top:8px;">
                                <p style="
                                    margin:0;
                                    color:#94a3b8;
                                    font-size:13px;
                                ">
                                    If you didn’t request this, please ignore this email.
                                </p>
                            </td>
                        </tr>

                        <!-- FOOTER -->
                        <tr>
                            <td align="center" style="padding-top:32px;">
                                <p style="
                                    margin:0;
                                    font-size:12px;
                                    color:#64748b;
                                ">
                                    © {datetime.datetime.utcnow().year} PolicyNav · AI Public Policy Assistant
                                </p>
                            </td>
                        </tr>

                    </table>
                </td>
            </tr>
        </table>
    </body>
    </html>
    '''

    msg.set_content("Your PolicyNav OTP is: " + otp)  # fallback
    msg.add_alternative(html_content, subtype="html")

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(EMAIL_ID, EMAIL_APP_PASSWORD)
            server.send_message(msg)
    except Exception as e:
        st.error("Failed to send OTP email. Please try again.")
        print("SMTP Error:", e)
        return False

    return True


def initiate_otp(email):
    user = get_user_by_email(email)
    if not user and st.session_state.get("otp_context") == "login":
        st.error("Email not registered")
        return

    conn = get_db_connection()
    cursor = conn.cursor()

    # Rate limit: allow OTP only once per 30 seconds
    cursor.execute(
        "SELECT created_at FROM otp_codes WHERE email = ?",
        (email,)
    )
    existing = cursor.fetchone()

    if existing and time.time() - existing["created_at"] < 30:
        conn.close()
        st.warning("Please wait 30 seconds before requesting another OTP.")
        return

    # Generate OTP
    otp = generate_otp()
    otp_hash = bcrypt.hashpw(otp.encode(), bcrypt.gensalt()).decode()
    created_at = time.time()
    expires_at = created_at + 120  # 2 minutes

    # Remove old OTPs
    cursor.execute("DELETE FROM otp_codes WHERE email = ?", (email,))

    #Store new OTP
    cursor.execute('''
        INSERT INTO otp_codes (email, otp_hash, expires_at, created_at)
        VALUES (?, ?, ?, ?)
    ''', (email, otp_hash, expires_at, created_at))

    conn.commit()
    conn.close()

    #Send OTP
    success = send_otp(email, otp)

    if not success:
        delete_otp(email)
        st.stop()  # prevents OTP verification from continuing

def get_otp(email):
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute('''
        SELECT otp_hash, expires_at, attempts FROM otp_codes
        WHERE email = ?
    ''', (email,))

    row = cursor.fetchone()
    conn.close()
    return dict(row) if row else None


def delete_otp(email):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM otp_codes WHERE email = ?", (email,))
    conn.commit()
    conn.close()

def update_last_login(email):
    conn = get_db_connection()
    conn.execute(
        "UPDATE users SET last_login=? WHERE email=?",
        (datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"), email)
    )
    conn.commit()
    conn.close()

def get_user_by_email(email):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE email = ?", (email,))
    user = cursor.fetchone()
    conn.close()
    return dict(user) if user else None

def increment_failed_attempts(email):
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute('''
        UPDATE users
        SET failed_attempts = failed_attempts + 1
        WHERE email = ?
    ''', (email,))

    conn.commit()
    conn.close()


def reset_failed_attempts(email):
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute('''
        UPDATE users
        SET failed_attempts = 0, lock_until = NULL
        WHERE email = ?
    ''', (email,))

    conn.commit()
    conn.close()


def lock_account(email, minutes=5):
    lock_time = time.time() + (minutes * 60)

    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute('''
        UPDATE users
        SET lock_until = ?
        WHERE email = ?
    ''', (lock_time, email))

    conn.commit()
    conn.close()

def get_user_by_username(username):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE username = ?", (username,))
    user = cursor.fetchone()
    conn.close()
    return dict(user) if user else None


def create_user(user_data):
    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        hashed_password = hash_password(user_data["password"])
        cursor.execute('''
            INSERT INTO users (email, username, password, security_question, security_answer)
            VALUES (?, ?, ?, ?, ?)
        ''', (
            user_data["email"],
            user_data["username"],
            hashed_password,
            user_data["security_question"],
            user_data["security_answer"]
        ))

        user_id = cursor.lastrowid

        cursor.execute('''
            INSERT INTO password_history (user_id, password, changed_at)
            VALUES (?, ?, ?)
        ''', (user_id, hashed_password, time.time()))

        conn.commit()

    except sqlite3.IntegrityError:
        conn.close()
        return False

    conn.close()
    return True

def is_password_reused(user_id, new_password):
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute('''
        SELECT password FROM password_history
        WHERE user_id = ?
    ''', (user_id,))

    rows = cursor.fetchall()
    conn.close()

    for row in rows:
        if verify_password(new_password, row["password"]):
            return True
    return False


def update_password(email, new_password):
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT id FROM users WHERE email = ?", (email,))
    user = cursor.fetchone()

    if not user:
        conn.close()
        return False

    user_id = user["id"]

    hashed_password = hash_password(new_password)

    # Update users table
    cursor.execute(
        "UPDATE users SET password = ? WHERE email = ?",
        (hashed_password, email)
    )

    # Insert into password history
    cursor.execute('''
        INSERT INTO password_history (user_id, password, changed_at)
        VALUES (?, ?, ?)
    ''', (user_id, hashed_password, time.time()))

    conn.commit()
    conn.close()
    return True




# --- JWT Utils ---
def create_access_token(data: dict):
    to_encode = data.copy()
    expire = datetime.datetime.utcnow() + datetime.timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    to_encode.update({
        "exp": expire,
        "iat": datetime.datetime.utcnow(),
        "iss": "PolicyNav",
        "aud": "PolicyNavUsers"
    })
    encoded_jwt = jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
    return encoded_jwt
def verify_token(token: str):
    try:
        payload = jwt.decode(
            token,
            SECRET_KEY,
            algorithms=[ALGORITHM],
            audience="PolicyNavUsers",
            issuer="PolicyNav"
        )
        return payload
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None

def require_admin():
    token = st.session_state.get("jwt_token")
    payload = verify_token(token)

    if not payload or payload.get("role") != "admin":
        st.error("Unauthorized access")
        st.stop()


# --- Format timestamps safely ---
def format_timestamp(ts):
    if ts:
        try:
            if isinstance(ts, str):
                ts = datetime.datetime.fromisoformat(ts)
            return ts.strftime("%Y-%m-%d %H:%M")
        except Exception:
            return str(ts)
    return "-"

# --- Validation Utils ---



def password_strength(password):
    # Block special characters visually
    if not password.isalnum():
        return 0

    score = 0

    if len(password) >= 8:
        score += 25
    if re.search(r"[A-Z]", password):
        score += 25
    if re.search(r"[a-z]", password):
        score += 25
    if re.search(r"[0-9]", password):
        score += 25

    return score


def check_password_strength(password: str):
    feedback = []

    if not password.isalnum():
        feedback.append("Password must contain only letters and numbers (no special characters).")

    if len(password) < 8:
        feedback.append("At least 8 characters")

    if not re.search(r"[A-Z]", password):
        feedback.append("At least one uppercase letter")

    if not re.search(r"[a-z]", password):
        feedback.append("At least one lowercase letter")

    if not re.search(r"[0-9]", password):
        feedback.append("At least one number")

    if feedback:
        return False, "Weak", feedback

    return True, "Strong", []



def verify_password(plain_password: str, hashed_password: str) -> bool:
    return bcrypt.checkpw(
        plain_password.encode(),
        hashed_password.encode()
    )

def is_valid_email(email):
    # Regex for standard email format
    pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    try:
        if re.match(pattern, email):
            return True
    except:
        return False
    return False

def logout():
    st.session_state.clear()
    st.session_state["page"] = "login"
    st.rerun()

# ---------- Readability Utilities ----------
def readability_metrics(text):
    sentences = max(1, text.count("."))
    words = len(text.split())
    avg_sentence_length = words / sentences

    # Simple Flesch-style heuristic (safe for exam)
    score = 206.835 - (1.015 * avg_sentence_length)

    return {
        "sentences": sentences,
        "words": words,
        "avg_sentence_length": round(avg_sentence_length, 2),
        "score": round(score, 2)
    }

# --- Session State Management ---
if 'jwt_token' not in st.session_state:
    st.session_state['jwt_token'] = None
if 'page' not in st.session_state:
    st.session_state['page'] = 'login'

# --- Styling ---
st.set_page_config(page_title="PolicyNav- AI Public Policy Assistant", page_icon="🤖", layout="wide")

st.markdown('''
    <style>
        /* ===== Global App ===== */
        .stApp {
            background: radial-gradient(circle at top, #0f172a 0%, #020617 60%);
            font-family: 'Inter', 'Segoe UI', sans-serif;
            color: #e5e7eb;
        }

        /* Remove Streamlit padding */
        .main > div {
            padding-top: 2rem;
        }

        /* ===== Headings ===== */
        h1 {
            text-align: center;
            color: #38bdf8;
            font-weight: 700;
            letter-spacing: 0.5px;
        }

        h4 {
            text-align: center;
            font-weight: 400;
        }

        /* ===== Buttons ===== */
        .stButton > button {
            width: 100%;
            height: 3em;
            border-radius: 10px;
            background: linear-gradient(135deg, #38bdf8, #6366f1);
            color: white;
            font-weight: 600;
            border: none;
            transition: all 0.25s ease;
        }

        /* Center buttons inside columns */
        div[data-testid="column"] > div:has(.stButton) {
            display: flex;
            justify-content: center;
        }


        .stButton > button:hover {
            transform: translateY(-2px);
            box-shadow: 0 8px 24px rgba(99,102,241,0.4);
        }

        /* ===== Inputs ===== */
        input, textarea {
            border-radius: 8px !important;
            background-color: #020617 !important;
            color: #e5e7eb !important;
        }

        /* ===== Sidebar ===== */
        section[data-testid="stSidebar"] {
            background: linear-gradient(180deg, #020617, #020617);
            border-right: 1px solid #1e293b;
        }

        section[data-testid="stSidebar"] h1 {
            color: #38bdf8;
        }

        /* ===== Chat Bubbles ===== */

          /* ===== Chat Bubbles ===== */

                  /* USER MESSAGE — Bright, active */
                  .user-msg {
                      text-align: right;
                      background: linear-gradient(135deg, #6366f1, #8b5cf6);
                      color: #ffffff;
                      padding: 12px 16px;
                      border-radius: 18px 18px 4px 18px;
                      margin: 10px 0;
                      display: inline-block;
                      max-width: 75%;
                      float: right;
                      clear: both;
                      font-size: 0.95rem;
                      box-shadow: 0 8px 26px rgba(99,102,241,0.45);
                      animation: fadeIn 0.25s ease-in;
                  }

                  /* BOT MESSAGE — Dark glass + cyan accent */
                  .bot-msg {
                      text-align: left;
                      background: rgba(15, 23, 42, 0.85);
                      backdrop-filter: blur(10px);
                      -webkit-backdrop-filter: blur(10px);
                      color: #e5e7eb;
                      padding: 12px 16px;
                      border-radius: 18px 18px 18px 4px;
                      margin: 10px 0;
                      display: inline-block;
                      max-width: 75%;
                      float: left;
                      clear: both;
                      font-size: 0.95rem;
                      border-left: 3px solid #38bdf8;
                      box-shadow: 0 10px 30px rgba(2,6,23,0.6);
                      animation: fadeIn 0.25s ease-in;
                  }



        /* ===== Animations ===== */
        @keyframes fadeIn {
            from {
                opacity: 0;
                transform: translateY(6px);
            }
            to {
                opacity: 1;
                transform: translateY(0);
            }
        }

        .welcome-container {
              text-align: center;
              margin-top: 20px;
              margin-bottom: 30px;
          }

          .welcome-text {
              font-family: 'Space Grotesk', 'Inter', sans-serif;
              font-size: 3rem;
              font-weight: 800;
              background: linear-gradient(90deg, #38bdf8, #6366f1, #22d3ee);
              -webkit-background-clip: text;
              -webkit-text-fill-color: transparent;
              letter-spacing: -1px;
          }

          .welcome-subtext {
              font-family: 'Inter', sans-serif;
              font-size: 1.1rem;
              color: #94a3b8;
              margin-top: 6px;
          }

        </style>
        ''', unsafe_allow_html=True)

# --- PAGES ---
def login_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([2, 3, 2])

    with col2:
        st.markdown(
            """
            <h1 style="
                text-align: center;
                font-size: 3rem;
                font-weight: 800;
                color: #2563eb;
                margin-bottom: 0.2em;
            ">
            PolicyNav – Public Policy Navigation Using AI
            </h1>
            """,
            unsafe_allow_html=True
        )

        st.markdown(
            "<h4 style='text-align:center; color:#94a3b8;'>Please sign in to continue</h4>",
            unsafe_allow_html=True
        )

        with st.form("login_form"):
            email = st.text_input("Email Address")
            password = st.text_input("Password", type="password")
            submitted = st.form_submit_button("Sign In")

            # ----- FORM SUBMISSION -----
            if submitted:
                # ----- FIELD VALIDATION -----
                if not email:
                    st.error("Email is required")
                    return
                if not is_valid_email(email):
                    st.error("Invalid email format (e.g. user@domain.com)")
                    return
                if not password:
                    st.error("Password is required")
                    return

                # ----- FETCH USER -----
                user = get_user_by_email(email)

                if not user:
                    st.error("No account found with this email")
                    return

                # 🔒 ----- ACCOUNT LOCK CHECK (CRITICAL FIX) -----
                if user.get("lock_until") and time.time() < user["lock_until"]:
                    remaining = int((user["lock_until"] - time.time()) / 60) + 1
                    st.error(f"Account locked. Try again in {remaining} minute(s).")
                    return

                # ----- PASSWORD VERIFICATION -----
                if not verify_password(password, user["password"]):
                    increment_failed_attempts(email)
                    user = get_user_by_email(email)  # refresh values

                    if user["failed_attempts"] >= 3:
                        lock_account(email)
                        st.error(
                            "Account locked due to 3 failed login attempts. "
                            "Try again after 5 minutes."
                        )
                    else:
                        remaining = 3 - user["failed_attempts"]
                        st.error(f"Incorrect password. {remaining} attempt(s) remaining.")
                    return

                # ✅ ----- SUCCESSFUL LOGIN -----
                reset_failed_attempts(email)
                update_last_login(email)

                token = create_access_token({
                    "sub": user["email"],
                    "username": user["username"],
                    "role": user["role"]
                })

                st.session_state["jwt_token"] = token
                st.success("Login successful!")
                time.sleep(0.5)

                # ----- ROLE-BASED REDIRECT -----
                if user.get("role") == "admin":
                    st.session_state["page"] = "admin_dashboard"
                else:
                    st.session_state["page"] = "dashboard"

                st.rerun()

        # ----- FOOTER ACTIONS -----
        st.markdown("---")
        c1, c2 = st.columns(2)

        with c1:
            if st.button("Forgot Password?", use_container_width=True):
                st.session_state["page"] = "forgot"
                st.rerun()

        with c2:
            if st.button("Create an Account", use_container_width=True):
                st.session_state["page"] = "signup"
                st.rerun()

def signup_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([2, 3, 2])

    with col2:
        st.title("Create Account")

        with st.form("signup_form"):
            username = st.text_input("Username (Required)").strip()
            email = st.text_input("Email Address (@domain.com required)").strip().lower()
            password = st.text_input(
                "Password",
                type="password",
                help="Minimum 8 chars, uppercase, lowercase, number"
            ).strip()


            if password:
                strength_percent = password_strength(password)  # returns 0–100

                st.progress(strength_percent / 100)

                if strength_percent < 40:
                    st.error("Weak password")
                elif strength_percent < 70:
                    st.warning("Moderate password")
                else:
                    st.success("Strong password")


            confirm_password = st.text_input("Confirm Password", type="password").strip()

            security_question = st.selectbox(
                "Security Question",
                [
                    "What is your pet name?",
                    "What is your college name?",
                    "Who is your favorite teacher?"
                ]
            )

            security_answer = st.text_input("Security Answer").strip()
            hashed_sa = bcrypt.hashpw(security_answer.lower().encode(), bcrypt.gensalt(rounds=12)).decode()
            submitted = st.form_submit_button("Sign Up")

            if submitted:
                errors = []

                # Username validation
                if not username:
                    errors.append("Username is mandatory.")
                elif get_user_by_username(username):
                    errors.append(f"Username '{username}' is already taken.")

                # Email validation
                if not email:
                    errors.append("Email is mandatory.")
                elif not is_valid_email(email):
                    errors.append("Invalid Email format (e.g. user@domain.com).")
                elif get_user_by_email(email):
                    errors.append(f"Email '{email}' is already registered.")

                # Password validation
                if not password:
                    errors.append("Password is mandatory.")

                is_valid, _, feedback = check_password_strength(password)

                if not is_valid:
                    errors.append("Password does not meet security requirements:")
                    for rule in feedback:
                        errors.append(f"• {rule}")


                # Confirm password
                if password != confirm_password:
                    errors.append("Passwords do not match.")



                # Security answer
                if not security_answer:
                    errors.append("Security answer is mandatory.")

                if errors:
                    for error in errors:
                        st.error(error)
                else:
                    user_data = {
                        "username": username,
                        "email": email,
                        "password": password,
                        "security_question": security_question,
                        "security_answer": hashed_sa,
                        "created_at": datetime.datetime.utcnow()
                    }

                    success = create_user(user_data)

                    if not success:
                        st.error("Email or username already exists.")
                    else:
                        token = create_access_token({
                            "sub": email,
                            "username": username
                        })

                        st.session_state['jwt_token'] = token
                        st.success("Account created successfully!")
                        time.sleep(1)
                        st.rerun()




        st.markdown("---")
        if st.button("Back to Login"):
            st.session_state['page'] = 'login'
            st.rerun()

def forgot_password_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])

    with col2:
        st.title("Forgot Password")

        email = st.text_input("Registered Email")

        if st.button("Verify Email"):
          user = get_user_by_email(email)

          if user:
              st.session_state['fp_email'] = email
              st.session_state['fp_question'] = user['security_question']
              st.session_state['page'] = 'forgot_verify'
              st.rerun()
          else:
              st.error("Email not found")

def forgot_verify_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])

    with col2:
        st.title("Security Verification")

        st.info(st.session_state['fp_question'])
        answer = st.text_input("Your Answer").strip()

        if st.button("Verify Answer"):
            user = get_user_by_email(st.session_state['fp_email'])

            if bcrypt.checkpw(answer.lower().encode(), user["security_answer"].encode()):
                st.session_state["otp_email"] = user["email"]
                st.session_state["otp_context"] = "forgot"

                initiate_otp(user["email"])

                st.session_state["page"] = "otp"
                st.rerun()
            else:
                st.error("Incorrect security answer")



def reset_password_page():
    token = st.session_state.get('reset_token')
    payload = verify_token(token)

    if not payload or payload.get("purpose") != "password_reset":
        st.error("Invalid or expired password reset session.")
        st.session_state['page'] = 'login'
        st.stop()

    email = payload['email']

    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 2, 1])

    with col2:
        st.title("Reset Password")

        new_password = st.text_input("New Password", type="password")
        confirm_password = st.text_input("Confirm Password", type="password")

        is_valid, _, feedback = check_password_strength(new_password)

        if st.button("Reset Password"):
            if new_password != confirm_password:
                st.error("Passwords do not match")
            elif not is_valid:
                  st.error("Password does not meet security requirements:")
                  for rule in feedback:
                      st.write(f"• {rule}")
                  st.stop()
            else:
                user = get_user_by_email(email)

                if not user:
                    st.error("User not found")

                elif is_password_reused(user["id"], new_password):
                    st.error("You cannot reuse an old password. Please choose a new one.")

                else:
                    update_password(payload["email"], new_password)

                    st.success("Password reset successful! Please login.")
                    st.session_state['reset_token'] = None
                    time.sleep(1)
                    st.session_state['page'] = 'login'
                    st.rerun()


def admin_dashboard():
    require_admin()  # Only admins can access

    st.markdown("<h1>Admin Dashboard</h1>", unsafe_allow_html=True)

    # --- Fetch user stats ---
    conn = get_db_connection()
    cursor = conn.cursor()

    total_users = cursor.execute("SELECT COUNT(*) FROM users").fetchone()[0]
    locked_users = cursor.execute(
        "SELECT COUNT(*) FROM users WHERE lock_until IS NOT NULL AND lock_until > ?",
        (time.time(),)
    ).fetchone()[0]

    conn.close()

    # --- Show overall metrics ---
    c1, c2 = st.columns(2)
    c1.metric("👥 Total Users", total_users)
    c2.metric("🔒 Locked Accounts", locked_users)

    st.markdown("---")
    st.subheader("User Management")

    # --- Fetch all users ---
    conn = get_db_connection()
    cursor = conn.cursor()
    users = cursor.execute(
        "SELECT email, username, role, lock_until, created_at, last_login FROM users ORDER BY role DESC, username ASC"
    ).fetchall()
    conn.close()

    # --- Table Headers ---
    cols = st.columns([3, 2, 1, 1, 2, 2, 4])  # Adjust widths
    cols[0].markdown("**Email**")
    cols[1].markdown("**Username**")
    cols[2].markdown("**Role**")
    cols[3].markdown("**Status**")
    cols[4].markdown("**Created At**")
    cols[5].markdown("**Last Login**")
    cols[6].markdown("**Actions**")

    # --- Table Rows ---
    for u in users:
        row_cols = st.columns([4, 2, 1, 1, 2, 2, 5])
        row_cols[0].write(u["email"])
        row_cols[1].write(u["username"])
        row_cols[2].write(u["role"])

        # Determine account status
        locked = u["lock_until"] and time.time() < u["lock_until"]
        status_text = f"<span style='color:red'>Locked</span>" if locked else f"<span style='color:green'>Active</span>"
        row_cols[3].markdown(status_text, unsafe_allow_html=True)

        # Format timestamps
        created = u["created_at"]
        last_login = u["last_login"]
        row_cols[4].write(format_timestamp(created))
        row_cols[5].write(format_timestamp(last_login))

        # --- Actions (non-admin users only) ---
        if u["role"] != "admin":
            action_cols = row_cols[6].columns(3)  # Side-by-side buttons: Unlock, Promote, Delete

            # Unlock account if locked
            if locked and action_cols[0].button("Unlock", key=f"unlock_{u['email']}"):
                reset_failed_attempts(u["email"])
                st.success(f"Account `{u['email']}` unlocked")
                st.session_state["page"] = "admin_dashboard"
                st.rerun()

            # Promote to admin
            if action_cols[1].button("Promote", key=f"promote_{u['email']}"):
                conn = get_db_connection()
                conn.execute("UPDATE users SET role='admin' WHERE email=?", (u["email"],))
                conn.commit()
                conn.close()
                st.success(f"User `{u['email']}` promoted to admin")
                st.session_state["page"] = "admin_dashboard"
                st.rerun()

            # Delete user
            if action_cols[2].button("Delete", key=f"delete_{u['email']}"):
                conn = get_db_connection()
                conn.execute("DELETE FROM users WHERE email=?", (u["email"],))
                conn.commit()
                conn.close()
                st.success(f"User `{u['email']}` deleted")
                st.session_state["page"] = "admin_dashboard"
                st.rerun()



def create_gauge(value, title, min_val, max_val, color):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        title={'text': title, 'font': {'size': 16}, 'align': 'center'},  # slightly bigger, centered
        gauge={
            'axis': {'range': [min_val, max_val], 'tickfont': {'size': 12}},
            'bar': {'color': color},
            'borderwidth': 2,
            'bordercolor': "gray",
        }
    ))

    fig.update_layout(
        height=350,          # increase height to avoid clipping
        margin=dict(l=10, r=10, t=60, b=20),  # push content down from top
    )
    return fig

# ---------- Readability Page ----------
def readability_page():
    # Make sure user is logged in
    if not st.session_state.get('user'):
        from streamlit_extras.switch_page_button import switch_page
        switch_page('login')
        return

    st.title("📖 Text Readability Analyzer")

    # --- Input Tabs ---
    tab1, tab2 = st.tabs(["✍️ Input Text", "📂 Upload File (TXT/PDF)"])
    text_input = ""

    with tab1:
        raw_text = st.text_area("Enter text to analyze (min 50 chars):", height=200)
        if raw_text:
            text_input = raw_text

    with tab2:
        uploaded_file = st.file_uploader("Upload a file", type=["txt", "pdf"])
        if uploaded_file:
            try:
                if uploaded_file.type == "application/pdf":
                    reader = PyPDF2.PdfReader(uploaded_file)
                    text = ""
                    for page in reader.pages:
                        text += page.extract_text() + "\n"
                    text_input = text
                    st.info(f"✅ Loaded {len(reader.pages)} pages from PDF.")
                else:
                    text_input = uploaded_file.read().decode("utf-8")
                    st.info(f"✅ Loaded TXT file: {uploaded_file.name}")
            except Exception as e:
                st.error(f"Error reading file: {e}")

    # --- Analyze Button ---
    if st.button("Analyze Readability"):
        if len(text_input) < 50:
            st.error("Text is too short (min 50 chars). Please enter more text or upload a valid file.")
        else:
            with st.spinner("Calculating advanced metrics..."):
                analyzer = ReadabilityAnalyzer(text_input)
                score = analyzer.get_all_metrics()

            # --- Overall Grade ---
            avg_grade = (score['Flesch-Kincaid Grade'] + score['Gunning Fog'] +
                         score['SMOG Index'] + score['Coleman-Liau']) / 4

            if avg_grade <= 6:
                level, color = "Beginner (Elementary)", "#28a745"
            elif avg_grade <= 10:
                level, color = "Intermediate (Middle School)", "#17a2b8"
            elif avg_grade <= 14:
                level, color = "Advanced (High School/College)", "#ffc107"
            else:
                level, color = "Expert (Professional/Academic)", "#dc3545"

            st.markdown(f"""
            <div style="background-color: #1f2937; padding: 20px; border-radius: 10px; border-left: 5px solid {color}; text-align: center;">
                <h2 style="margin:0; color: {color} !important;">Overall Level: {level}</h2>
                <p style="margin:5px 0 0 0; color: #9ca3af;">Approximate Grade Level: {int(avg_grade)}</p>
            </div>
            """, unsafe_allow_html=True)

            # --- Gauges ---
            st.markdown("### 📈 Detailed Metrics")
            c1, c2, c3 = st.columns(3)
            c1.plotly_chart(create_gauge(score["Flesch Reading Ease"], "Flesch Reading Ease", 0, 100, "#00ffcc"), use_container_width=True)
            c2.plotly_chart(create_gauge(score["Flesch-Kincaid Grade"], "Flesch-Kincaid Grade", 0, 20, "#ff00ff"), use_container_width=True)
            c3.plotly_chart(create_gauge(score["SMOG Index"], "SMOG Index", 0, 20, "#ffff00"), use_container_width=True)

            c4, c5 = st.columns(2)
            c4.plotly_chart(create_gauge(score["Gunning Fog"], "Gunning Fog", 0, 20, "#00ccff"), use_container_width=True)
            c5.plotly_chart(create_gauge(score["Coleman-Liau"], "Coleman-Liau", 0, 20, "#ff9900"), use_container_width=True)

            st.markdown("### ℹ️ Understanding the Metrics")

            with st.expander("📘 What is Flesch Reading Ease?"):
                st.markdown("""
                - Measures how easy a text is to read.
                - Higher score = easier to understand.
                - **90–100**: Very easy
                - **60–70**: Standard
                - **0–30**: Very difficult
                """)

            with st.expander("📗 What is Flesch-Kincaid Grade Level?"):
                st.markdown("""
                - Indicates the **school grade level** required to understand the text.
                - Example: A score of **8** means an 8th-grade student can understand it.
                """)

            with st.expander("📕 What is Gunning Fog Index?"):
                st.markdown("""
                - Estimates the years of formal education needed to understand the text.
                - Penalizes long sentences and complex words.
                """)

            with st.expander("📙 What is SMOG Index?"):
                st.markdown("""
                - Best suited for longer texts.
                - Focuses on words with **3 or more syllables**.
                - Commonly used in academic and policy documents.
                """)

            with st.expander("📓 What is Coleman–Liau Index?"):
                st.markdown("""
                - Uses **characters per word** instead of syllables.
                - Works well for digital and typed content.
                """)

            # --- Text Stats ---
            st.markdown("### 📝 Text Statistics")
            s1, s2, s3, s4, s5 = st.columns(5)
            s1.metric("Sentences", analyzer.num_sentences)
            s2.metric("Words", analyzer.num_words)
            s3.metric("Syllables", analyzer.syllables)
            s4.metric("Complex Words", analyzer.complex_words)
            s5.metric("Characters", analyzer.char_count)

def dashboard_page():
    token   = st.session_state.get('jwt_token')
    payload = verify_token(token)

    if not payload:
        st.session_state['jwt_token'] = None
        st.session_state['page'] = 'login'
        st.warning("Session expired or invalid. Please login again.")
        time.sleep(1)
        st.rerun()
        return

    username = payload.get("username", "User")
    user_email = payload.get("sub", "")

    # ── Init DB tables for history/feedback ────────────────────────────────
    conn = get_db_connection()
    cur  = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS qa_history (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            email       TEXT NOT NULL,
            question    TEXT NOT NULL,
            answer      TEXT NOT NULL,
            sources     TEXT,
            language    TEXT DEFAULT 'English',
            elapsed     REAL,
            timestamp   TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS qa_feedback (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            history_id  INTEGER,
            email       TEXT NOT NULL,
            question    TEXT NOT NULL,
            rating      INTEGER NOT NULL,   -- 1=thumbs up, 0=thumbs down
            comment     TEXT DEFAULT '',
            timestamp   TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

    # ── Page Header ────────────────────────────────────────────────────────
    st.markdown(f'''
        <div class="welcome-container">
            <div class="welcome-text">Welcome, {username}</div>
            <div class="welcome-subtext">Ask me anything about public policy</div>
        </div>
    ''', unsafe_allow_html=True)

    # ── Tabs: Chat | History ───────────────────────────────────────────────
    tab_chat, tab_history = st.tabs(["💬 Chat", "📜 History"])

    # ══════════════════════════════════════════════════════════════════════
    # TAB 1 — CHAT
    # ══════════════════════════════════════════════════════════════════════
    with tab_chat:
        col_lang, col_simp, col_clear = st.columns([3, 2, 2])
        with col_lang:
            target_lang = st.selectbox(
                "Response language",
                ["English", "Hindi", "Tamil", "Kannada",
                 "Telugu", "Marathi", "Bengali", "Malayalam"],
                key="chat_lang"
            )
        with col_simp:
            simplify = st.toggle("🧠 Simplify language", key="chat_simplify")
        with col_clear:
            if st.button("🗑️ Clear chat", use_container_width=True):
                st.session_state["chat_history"] = []
                st.rerun()

        st.markdown("---")

        if "chat_history" not in st.session_state:
            st.session_state["chat_history"] = []

        # ── Render existing messages ───────────────────────────────────────
        for i, msg in enumerate(st.session_state["chat_history"]):
            if msg["role"] == "user":
                st.markdown(
                    f'<div class="user-msg">{msg["content"]}</div>',
                    unsafe_allow_html=True
                )
            else:
                st.markdown(
                    f'<div class="bot-msg">{msg["content"]}</div>',
                    unsafe_allow_html=True
                )

                # ── Source documents expander ──────────────────────────────
                if msg.get("sources"):
                    with st.expander(f"📚 Sources ({len(msg['sources'])} documents)", expanded=False):
                        for j, src_doc in enumerate(msg["sources"]):
                            st.markdown(
                                f"""
                                <div style="background:#0f172a;border:1px solid #1e293b;
                                            border-radius:10px;padding:12px;margin-bottom:8px;">
                                    <div style="color:#38bdf8;font-weight:600;font-size:0.85rem;
                                                margin-bottom:6px;">
                                        📄 {src_doc['filename']}
                                    </div>
                                    <div style="color:#94a3b8;font-size:0.82rem;
                                                line-height:1.5;white-space:pre-wrap;">
                                        {src_doc['content'][:400]}{'…' if len(src_doc['content']) > 400 else ''}
                                    </div>
                                </div>
                                """,
                                unsafe_allow_html=True
                            )

                # ── Feedback row ───────────────────────────────────────────
                msg_id   = msg.get("id", i)
                fb_key   = f"feedback_given_{msg_id}"
                if not st.session_state.get(fb_key):
                    fb_cols = st.columns([1, 1, 8])
                    with fb_cols[0]:
                        if st.button("👍", key=f"up_{msg_id}", help="Good answer"):
                            _save_feedback(user_email, msg, rating=1)
                            st.session_state[fb_key] = "up"
                            st.toast("Thanks for the feedback! 🎉")
                    with fb_cols[1]:
                        if st.button("👎", key=f"dn_{msg_id}", help="Poor answer"):
                            _save_feedback(user_email, msg, rating=0)
                            st.session_state[fb_key] = "down"
                            st.toast("Thanks — we'll improve! 🔧")
                else:
                    given = st.session_state[fb_key]
                    emoji = "👍 Helpful" if given == "up" else "👎 Not helpful"
                    st.markdown(
                        f"<small style='color:#64748b'>{emoji}</small>",
                        unsafe_allow_html=True
                    )

        # ── Input form ─────────────────────────────────────────────────────
        with st.form(key='chat_form', clear_on_submit=True):
            col1, col2 = st.columns([6, 1])
            with col1:
                user_input = st.text_input(
                    "Message PolicyNav...",
                    placeholder="Ask a policy question...",
                    label_visibility="collapsed"
                )
            with col2:
                submit_button = st.form_submit_button("Send")

        if submit_button and user_input.strip():
            st.session_state["chat_history"].append(
                {"role": "user", "content": user_input}
            )
            with st.spinner("Searching knowledge base..."):
                t_start = time.time()
                try:
                    answer, source_docs = nlp_engine.answer_policy_question(
                        native_question=user_input,
                        target_lang=target_lang,
                        simplify=simplify
                    )
                    elapsed = round(time.time() - t_start, 2)

                    if source_docs:
                        sources_str = ", ".join(
                            list(dict.fromkeys(d["filename"] for d in source_docs))
                        )
                        bot_display = (
                            f"{answer}<br><br>"
                            f"<small>📚 Sources: {sources_str} &nbsp;|&nbsp; ⏱ {elapsed}s</small>"
                        )
                    else:
                        source_docs  = []
                        sources_str  = ""
                        bot_display  = (
                            f"{answer}<br><br>"
                            "<small>⚠️ No matching documents found in knowledge base.</small>"
                        )

                    # persist to DB
                    history_id = _save_qa_history(
                        user_email, user_input, answer,
                        sources_str, target_lang, elapsed
                    )

                    bot_msg = {
                        "role":    "assistant",
                        "content": bot_display,
                        "sources": source_docs,
                        "answer_plain": answer,
                        "id":      history_id,
                        "question": user_input,
                    }

                except Exception as e:
                    bot_msg = {
                        "role":    "assistant",
                        "content": f"⚠️ Error: {e}",
                        "sources": [],
                        "id":      i if 'i' in dir() else 0,
                        "question": user_input,
                    }

            st.session_state["chat_history"].append(bot_msg)
            st.rerun()

     # ── Global Q&A History ──────────────────────────────────────────────────
    st.markdown("---")
    st.subheader("📋 Global Q&A History")

    conn = get_db_connection()
    cursor = conn.cursor()

    # Summary stats
    total_qa    = cursor.execute("SELECT COUNT(*) FROM qa_history").fetchone()[0]
    active_users = cursor.execute("SELECT COUNT(DISTINCT email) FROM qa_history").fetchone()[0]
    avg_elapsed = cursor.execute("SELECT AVG(elapsed) FROM qa_history").fetchone()[0]

    s1, s2, s3 = st.columns(3)
    s1.metric("💬 Total Questions", total_qa)
    s2.metric("👤 Active Users", active_users)
    s3.metric("⚡ Avg Response Time", f"{avg_elapsed:.1f}s" if avg_elapsed else "—")

    # Last 200 entries — brief table only (no full answers)
    rows = cursor.execute("""
        SELECT qh.timestamp, u.username, qh.email, qh.question, qh.language, qh.elapsed
        FROM qa_history qh
        LEFT JOIN users u ON qh.email = u.email
        ORDER BY qh.timestamp DESC
        LIMIT 200
    """).fetchall()
    conn.close()

    if rows:
        # Search/filter box
        qa_search = st.text_input("🔍 Filter by username or question", key="admin_qa_search", placeholder="Type to filter...")

        header_cols = st.columns([2, 2, 4, 1, 1])
        header_cols[0].markdown("**Timestamp**")
        header_cols[1].markdown("**User**")
        header_cols[2].markdown("**Question (preview)**")
        header_cols[3].markdown("**Lang**")
        header_cols[4].markdown("**Time (s)**")

        for row in rows:
            ts, username, email, question, language, elapsed = row
            username_display = username or email or "unknown"
            question_preview = (question[:80] + "…") if question and len(question) > 80 else (question or "")

            # Apply search filter
            if qa_search and qa_search.lower() not in username_display.lower() and qa_search.lower() not in question_preview.lower():
                continue

            r = st.columns([2, 2, 4, 1, 1])
            r[0].caption(format_timestamp(ts))
            r[1].write(username_display)
            r[2].write(question_preview)
            r[3].write(language or "EN")
            r[4].write(f"{elapsed:.1f}" if elapsed else "—")
    else:
        st.info("No Q&A history yet.")

    # ══════════════════════════════════════════════════════════════════════
    # TAB 2 — HISTORY
    # ══════════════════════════════════════════════════════════════════════
    with tab_history:
        st.markdown(
            "<h3 style='color:#38bdf8;'>📜 Your Q&A History</h3>",
            unsafe_allow_html=True
        )

        try:
            conn  = get_db_connection()
            cur   = conn.cursor()
            rows  = cur.execute("""
                SELECT id, question, answer, sources, language, elapsed, timestamp
                FROM qa_history
                WHERE email = ?
                ORDER BY timestamp DESC
                LIMIT 100
            """, (user_email,)).fetchall()
            conn.close()
        except Exception:
            rows = []

        if not rows:
            st.info("No history yet. Ask a question in the Chat tab!")
        else:
            # Stats bar
            h_col1, h_col2, h_col3 = st.columns(3)
            h_col1.metric("Total Questions", len(rows))
            avg_t = round(sum(r[5] or 0 for r in rows) / max(len(rows), 1), 2)
            h_col2.metric("Avg Response Time", f"{avg_t}s")
            langs = collections.Counter(r[4] for r in rows if r[4])
            top_lang = langs.most_common(1)[0][0] if langs else "—"
            h_col3.metric("Most Used Language", top_lang)

            st.markdown("---")

            # Search / filter
            search_q = st.text_input(
                "🔍 Search history",
                placeholder="Filter by keyword...",
                key="history_search"
            )

            for row in rows:
                hid, question, answer, sources, language, elapsed, ts = row
                if search_q and search_q.lower() not in question.lower() and \
                   search_q.lower() not in (answer or "").lower():
                    continue

                with st.expander(
                    f"🗓 {ts[:16]}  |  🌐 {language or 'English'}  |  ⏱ {elapsed or '—'}s  —  {question[:80]}{'…' if len(question)>80 else ''}",
                    expanded=False
                ):
                    st.markdown(
                        f"""
                        <div style="background:#0f172a;border:1px solid #1e293b;
                                    border-radius:12px;padding:16px;margin-bottom:8px;">
                            <div style="color:#a78bfa;font-weight:700;margin-bottom:8px;">
                                ❓ {question}
                            </div>
                            <div style="color:#e5e7eb;font-size:0.92rem;line-height:1.6;">
                                {answer or '—'}
                            </div>
                            {f'<div style="color:#64748b;font-size:0.8rem;margin-top:10px;">📚 {sources}</div>' if sources else ''}
                        </div>
                        """,
                        unsafe_allow_html=True
                    )

                    # Feedback for history items
                    fb_hist_key = f"hist_fb_{hid}"
                    if not st.session_state.get(fb_hist_key):
                        fc1, fc2, _ = st.columns([1, 1, 8])
                        with fc1:
                            if st.button("👍", key=f"h_up_{hid}"):
                                _save_feedback(user_email, {"question": question, "id": hid, "answer_plain": answer}, rating=1)
                                st.session_state[fb_hist_key] = "up"
                                st.toast("Thanks! 🎉")
                        with fc2:
                            if st.button("👎", key=f"h_dn_{hid}"):
                                _save_feedback(user_email, {"question": question, "id": hid, "answer_plain": answer}, rating=0)
                                st.session_state[fb_hist_key] = "down"
                                st.toast("Thanks for the feedback! 🔧")


# ── Feedback + History helpers (placed right before dashboard_page in app.py) ─

def _save_qa_history(email, question, answer, sources, language, elapsed):
    """Persist a Q&A exchange. Returns inserted row id."""
    try:
        conn = get_db_connection()
        cur  = conn.cursor()
        cur.execute("""
            INSERT INTO qa_history (email, question, answer, sources, language, elapsed)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (email, question, answer, sources, language, elapsed))
        conn.commit()
        row_id = cur.lastrowid
        conn.close()
        return row_id
    except Exception:
        return 0

def _save_feedback(email, msg, rating):
    """Persist a 👍/👎 feedback entry."""
    try:
        conn = get_db_connection()
        cur  = conn.cursor()
        cur.execute("""
            INSERT INTO qa_feedback (history_id, email, question, rating)
            VALUES (?, ?, ?, ?)
        """, (msg.get("id", 0), email, msg.get("question", ""), rating))
        conn.commit()
        conn.close()
    except Exception:
        pass



def summarization_page():
    token   = st.session_state.get('jwt_token')
    payload = verify_token(token)
    if not payload:
        st.session_state['page'] = 'login'
        st.rerun()
        return

    st.markdown('''
    <div style="background:linear-gradient(135deg,#020617,#0f172a);
                padding:2rem;border-radius:16px;margin-bottom:1.5rem;
                border:1px solid #1e293b;">
        <h1 style="color:white;margin:0;">📝 Multi-Language Summarization</h1>
        <p style="color:#94a3b8;margin:0.5rem 0 0 0;">
            Summarize any policy document in your preferred language
        </p>
    </div>
    ''', unsafe_allow_html=True)

    LANG_MAP = {
        "English":   "eng_Latn",
        "Hindi":     "hin_Deva",
        "Tamil":     "tam_Taml",
        "Kannada":   "kan_Knda",
        "Telugu":    "tel_Telu",
        "Marathi":   "mar_Deva",
        "Bengali":   "ben_Beng",
        "Malayalam": "mal_Mlym",
    }

    col1, col2 = st.columns([2, 1])
    with col1:
        target_lang = st.selectbox(
            "📌 Summary Language",
            list(LANG_MAP.keys()),
            key="sum_lang"
        )
    with col2:
        summary_length = st.select_slider(
            "📏 Summary Length",
            options=["Short", "Medium", "Long"],
            value="Medium"
        )

    length_tokens = {"Short": 80, "Medium": 180, "Long": 300}
    max_tokens = length_tokens[summary_length]

    input_method = st.radio(
        "Input Method",
        ["📝 Paste Text", "📁 Upload File"],
        horizontal=True,
        key="sum_input"
    )

    text_to_summarize = ""

    if "📝 Paste Text" in input_method:
        text_to_summarize = st.text_area(
            "Paste policy text here:",
            height=220,
            placeholder="Paste any public policy document, government scheme, or legal text...",
            key="sum_text"
        )
    else:
        uploaded = st.file_uploader(
            "Upload file (.txt, .pdf, .docx)",
            type=["txt", "pdf", "docx"],
            key="sum_file"
        )
        if uploaded:
            with st.spinner("Extracting text..."):
                try:
                    if uploaded.name.endswith(".txt"):
                        text_to_summarize = uploaded.read().decode("utf-8", errors="ignore")
                    elif uploaded.name.endswith(".pdf"):
                        reader = PyPDF2.PdfReader(io.BytesIO(uploaded.read()))
                        text_to_summarize = " ".join(
                            p.extract_text() or "" for p in reader.pages
                        )
                    elif uploaded.name.endswith(".docx"):
                        import docx as docxlib
                        doc = docxlib.Document(io.BytesIO(uploaded.read()))
                        text_to_summarize = " ".join(p.text for p in doc.paragraphs)
                except Exception as e:
                    st.error(f"Error extracting text: {e}")

            if text_to_summarize:
                st.success(f"✅ Extracted {len(text_to_summarize.split())} words")
                with st.expander("📄 Preview extracted text"):
                    st.write(text_to_summarize[:2000] + ("..." if len(text_to_summarize) > 2000 else ""))

    if text_to_summarize and len(text_to_summarize.split()) >= 30:
        if st.button("🔍 Summarize", use_container_width=True):
            with st.spinner("Generating summary via Qwen → NLLB pipeline... ⏳"):
                try:
                    # Sir's method: nlp_engine.summarize_document uses Qwen + NLLB
                    # Step 1: Generate English summary using Qwen
                    from nlp_engine import generate_english_response, translate_fast
                    eng_summary = generate_english_response(
                        f"Summarize this policy document into clear, concise bullet points "
                        f"(length: {summary_length}):\n\n{text_to_summarize[:3000]}"
                    )
                    # Step 2: Translate using NLLB if not English
                    final_summary = translate_fast(eng_summary, "English", target_lang)

                    st.session_state["last_summary"] = {
                        "english": eng_summary,
                        "translated": final_summary,
                        "lang": target_lang
                    }
                    st.session_state["_sum_orig_text"] = text_to_summarize
                except Exception as e:
                    st.error(f"⚠️ Summarization error: {e}")

    elif text_to_summarize:
        st.warning("⚠️ Please enter at least 30 words for summarization.")

    # Display results
    if st.session_state.get("last_summary"):
        summ = st.session_state["last_summary"]
        st.markdown("---")
        st.markdown("### 📋 Summary Results")

        if summ["lang"] != "English":
            tab1, tab2 = st.tabs([f"🌐 {summ['lang']} Summary", "🇬🇧 English Summary"])
            with tab1:
                st.markdown(
                    f'''<div style="background:#0f172a;border:1px solid #1e293b;
                                    border-radius:12px;padding:1.5rem;color:white;
                                    font-size:1.05rem;line-height:1.7;">
                        {summ["translated"]}
                    </div>''',
                    unsafe_allow_html=True
                )
            with tab2:
                st.markdown(
                    f'''<div style="background:#0f172a;border:1px solid #1e293b;
                                    border-radius:12px;padding:1.5rem;color:white;
                                    font-size:1.05rem;line-height:1.7;">
                        {summ["english"]}
                    </div>''',
                    unsafe_allow_html=True
                )
        else:
            st.markdown(
                f'''<div style="background:#0f172a;border:1px solid #1e293b;
                                border-radius:12px;padding:1.5rem;color:white;
                                font-size:1.05rem;line-height:1.7;">
                    {summ["english"]}
                </div>''',
                unsafe_allow_html=True
            )

        col1, col2 = st.columns(2)
        orig = st.session_state.get("_sum_orig_text", "")
        col1.metric("📝 Original Words", len(orig.split()) if orig else "N/A")
        col2.metric("📋 Summary Words", len(summ["english"].split()))

        if st.button("🗑 Clear Summary"):
            st.session_state.pop("last_summary", None)
            st.rerun()

def knowledge_graph_page():
    token   = st.session_state.get('jwt_token')
    payload = verify_token(token)
    if not payload:
        st.session_state['page'] = 'login'
        st.rerun()
        return

    import streamlit.components.v1 as components

    st.markdown('''
    <div style="background:linear-gradient(135deg,#020617,#0f172a);
                padding:2rem;border-radius:16px;margin-bottom:1.5rem;
                border:1px solid #1e293b;">
        <h1 style="color:white;margin:0;">🕸 Knowledge Graph</h1>
        <p style="color:#94a3b8;margin:0.5rem 0 0 0;">
            Rich entity extraction · Relationship mapping · Policy domain clustering
        </p>
    </div>
    ''', unsafe_allow_html=True)

    # ── Load docs from FAISS vector store (already ingested) ───────────────
    try:
        all_chunks = vector_store.get_all_documents()
    except Exception as e:
        st.error(f"Could not load documents from vector store: {e}")
        return

    if not all_chunks:
        st.warning("⚠️ No documents found in the knowledge base. Please run the PDF ingestion cell first.")
        return

    # Deduplicate chunks → one entry per unique filename with combined content
    seen = {}
    for chunk in all_chunks:
        fname = chunk["filename"]
        if fname not in seen:
            seen[fname] = chunk["content"]
        else:
            seen[fname] += " " + chunk["content"]

    docs = [{"filename": fname, "content": content} for fname, content in seen.items()]

    st.markdown(
        f'''<div style="background:#0f172a;border:1px solid #1e293b;border-radius:12px;
                    padding:1rem 1.5rem;margin-bottom:1rem;display:flex;gap:2rem;">
            <div>
                <div style="color:#94a3b8;font-size:0.8rem;">Documents in knowledge base</div>
                <div style="color:#38bdf8;font-size:1.6rem;font-weight:700;">{len(docs)}</div>
            </div>
            <div>
                <div style="color:#94a3b8;font-size:0.8rem;">Total chunks indexed</div>
                <div style="color:#22d3ee;font-size:1.6rem;font-weight:700;">{len(all_chunks):,}</div>
            </div>
        </div>''',
        unsafe_allow_html=True
    )

    # ── Controls row ───────────────────────────────────────────────────────
    ALL_ENTITY_TYPES = [
        "ORG", "LAW", "GPE", "PERSON",
        "MONEY", "PERCENT", "DATE", "NORP",
        "PRODUCT", "EVENT", "FACILITY",
    ]

    ctrl1, ctrl2 = st.columns([3, 2])
    with ctrl1:
        selected_types = st.multiselect(
            "🏷 Entity types to include",
            options=ALL_ENTITY_TYPES,
            default=["ORG", "LAW", "GPE", "PERSON", "NORP", "PRODUCT"],
            key="kg_entity_types"
        )
    with ctrl2:
        max_docs = st.slider(
            "📄 Max documents to process", 1, 60,
            min(len(docs), 20), key="kg_max_docs"
        )

    # ── Generate ───────────────────────────────────────────────────────────
    if st.button("🕸 Generate Knowledge Graph", use_container_width=True):
        if not selected_types:
            st.warning("Please select at least one entity type.")
            return

        with st.spinner("Extracting entities and relationships…"):
            try:
                graph_path, entity_freq, stats = knowledge_graph.generate_interactive_graph(
                    docs,
                    entity_types=selected_types,
                    max_docs=max_docs
                )
                if graph_path:
                    st.session_state["kg_path"]        = graph_path
                    st.session_state["kg_entity_freq"] = entity_freq
                    st.session_state["kg_stats"]       = stats
                    st.success("✅ Knowledge Graph generated!")
                else:
                    st.warning("⚠️ No entities found. Try documents with more named entities.")
            except Exception as e:
                st.error(f"⚠️ Error: {e}")

    # ── Render graph if available ──────────────────────────────────────────
    if st.session_state.get("kg_path"):
        # ── Stats dashboard ────────────────────────────────────────────────
        entity_freq = st.session_state.get("kg_entity_freq", {})
        stats       = st.session_state.get("kg_stats", {})

        st.markdown("### 📊 Graph Statistics")
        s1, s2, s3, s4 = st.columns(4)
        s1.metric("📄 Documents",  stats.get("document", 0))
        s2.metric("🏷 Entities",   stats.get("entity",   0))
        s3.metric("🔗 Total Nodes", sum(stats.values()))
        top_ents = sorted(entity_freq.items(), key=lambda x: -x[1])[:1]
        s4.metric("🔥 Top Entity", top_ents[0][0] if top_ents else "—")

        # ── Legend ─────────────────────────────────────────────────────────
        st.markdown("### 🎨 Legend")
        ENTITY_COLORS_DISPLAY = {
            "ORG":"#38bdf8","LAW":"#f59e0b","GPE":"#22d3ee",
            "PERSON":"#a78bfa","MONEY":"#34d399","NORP":"#f472b6",
            "PRODUCT":"#fb923c","DATE":"#94a3b8","FACILITY":"#60a5fa",
        }
        DOMAIN_COLORS_DISPLAY = {
            "Agriculture":"#86efac","Health":"#f9a8d4","Education":"#fde68a",
            "Housing":"#93c5fd","Finance":"#6ee7b7","Energy":"#fcd34d",
            "Labour":"#c4b5fd","Digital":"#67e8f9","Social":"#fca5a5",
        }
        leg_html = '<div style="display:flex;flex-wrap:wrap;gap:8px;margin-bottom:1rem;">'
        leg_html += '<span style="color:#94a3b8;font-size:0.8rem;width:100%;margin-bottom:2px;">Entity types:</span>'
        for etype, col in ENTITY_COLORS_DISPLAY.items():
            leg_html += (
                f'<span style="background:{col}22;border:1px solid {col};'
                f'color:{col};padding:3px 10px;border-radius:20px;'
                f'font-size:0.78rem;font-weight:600;">{etype}</span>'
            )
        leg_html += '</div>'
        leg_html += '<div style="display:flex;flex-wrap:wrap;gap:8px;margin-bottom:1rem;">'
        leg_html += '<span style="color:#94a3b8;font-size:0.8rem;width:100%;margin-bottom:2px;">Document domains (⭐ star nodes):</span>'
        for domain, col in DOMAIN_COLORS_DISPLAY.items():
            leg_html += (
                f'<span style="background:{col}33;border:1px solid {col};'
                f'color:{col};padding:3px 10px;border-radius:20px;'
                f'font-size:0.78rem;">{domain}</span>'
            )
        leg_html += '</div>'
        st.markdown(leg_html, unsafe_allow_html=True)

        # ── Top entities table ─────────────────────────────────────────────
        if entity_freq:
            with st.expander("🔥 Top 20 Most Frequent Entities", expanded=False):
                top20 = sorted(entity_freq.items(), key=lambda x: -x[1])[:20]
                rows_html = "".join(
                    f'<tr><td style="padding:6px 12px;color:#e5e7eb;">{ent}</td>'
                    f'<td style="padding:6px 12px;color:#38bdf8;font-weight:700;">{freq}</td></tr>'
                    for ent, freq in top20
                )
                st.markdown(
                    f'<table style="width:100%;border-collapse:collapse;">'
                    f'<tr><th style="text-align:left;color:#94a3b8;padding:6px 12px;">Entity</th>'
                    f'<th style="text-align:left;color:#94a3b8;padding:6px 12px;">Frequency</th></tr>'
                    f'{rows_html}</table>',
                    unsafe_allow_html=True
                )

        # ── Render HTML graph ──────────────────────────────────────────────
        st.markdown("### 🕸 Interactive Knowledge Graph")
        st.markdown(
            """
            <div style="background:#0f172a;padding:0.6rem 1rem;border-radius:8px;
                        color:#94a3b8;font-size:0.82rem;margin-bottom:0.5rem;">
                ⭐ Star nodes = Documents &nbsp;|&nbsp;
                Coloured circles = Entities &nbsp;|&nbsp;
                Edge labels = Relationship types &nbsp;|&nbsp;
                Drag · Zoom · Hover for details
            </div>
            """,
            unsafe_allow_html=True
        )

        try:
            with open(st.session_state["kg_path"], "r", encoding="utf-8") as f:
                html_data = f.read()
            components.html(html_data, height=700, scrolling=False)
        except Exception as e:
            st.error(f"Could not render graph: {e}")

        # ── Clear ──────────────────────────────────────────────────────────
        if st.button("🗑 Clear Graph"):
            for k in ["kg_path", "kg_entity_freq", "kg_stats"]:
                st.session_state.pop(k, None)
            st.rerun()



def sidebar_navigation():
    with st.sidebar:
        st.markdown(
            """
            <style>
            /* Sidebar header */
            .sidebar .sidebar-content {
                background: linear-gradient(180deg, #020617, #0f172a);
                padding-top: 2rem;
            }

            /* Logo / Title */
            .sidebar h3 {
                font-family: 'Space Grotesk', 'Inter', sans-serif;
                font-size: 1.8rem;
                font-weight: 800;
                text-align: center;
                background: linear-gradient(90deg,#38bdf8,#6366f1,#22d3ee);
                -webkit-background-clip: text;
                -webkit-text-fill-color: transparent;
                margin-bottom: 0.3rem;
            }

            /* User info */
            .sidebar p {
                font-family: 'Inter', sans-serif;
                text-align: center;
                color: #94a3b8;
                font-size: 0.95rem;
                margin-top: 0;
                margin-bottom: 1rem;
            }

            /* Option menu adjustments */
            [data-testid="stVerticalBlock"] > div > ul > li {
                font-family: 'Inter', sans-serif;
                font-size: 1rem;
                color: #cbd5e1;
                border-radius: 12px;
                margin-bottom: 4px;
                padding: 8px 12px;
                transition: all 0.3s ease;
            }


            [data-testid="stVerticalBlock"] > div > ul > li:hover {
                background: linear-gradient(135deg,#38bdf8,#6366f1);
                color: #020617 !important;
                transform: translateX(4px);
                box-shadow: 0 6px 16px rgba(56,189,248,0.5);
            }

            /* Selected menu item */
            [data-testid="stVerticalBlock"] > div > ul > li.active {
                background: linear-gradient(135deg,#6366f1,#22d3ee);
                color: #f8fafc !important;
                font-weight: 700;
                font-size: 1.05rem;
                box-shadow: 0 8px 20px rgba(56,189,248,0.6);
            }

            /* Logout button */
            .stButton > button {
                font-family: 'Inter', sans-serif;
                font-weight: 600;
                border-radius: 12px;
                background: linear-gradient(135deg,#6366f1,#22d3ee);
                color: #f8fafc;
                border: none;
                transition: all 0.25s ease;
            }
            .stButton > button:hover {
                transform: translateY(-2px);
                box-shadow: 0 8px 24px rgba(56,189,248,0.45);
            }
            </style>
            """,
            unsafe_allow_html=True
        )

        st.markdown(
            '''
            <h1 style="
                text-align:center;
                font-weight:800;
                font-size:3rem;
                background: linear-gradient(90deg, #38bdf8, #6366f1, #22d3ee);
                -webkit-background-clip: text;
                -webkit-text-fill-color: transparent;
            ">
                PolicyNav
            </h1>
            ''',
            unsafe_allow_html=True
        )

        st.markdown(
            f"<p>👤 {st.session_state.get('user')}</p>", unsafe_allow_html=True
        )

        st.markdown("---")

        menu_items = []
        icons = []

        if st.session_state.get("role") == "admin":
            menu_items = ["Admin"]
            icons = ["shield-lock"]
        else:
            menu_items = ["Chat", "Summarize", "Knowledge Graph", "Readability"]
            icons = ["chat-dots", "file-text", "diagram-3", "book"]

        selected = option_menu(
            menu_title=None,  # already have title
            options=menu_items,
            icons=icons,
            menu_icon="layers",
            default_index=0,
            styles={
                "container": {"padding": "0", "background-color": "#020617"},
                "icon": {"color": "#38bdf8", "font-size": "18px"},
                "nav-link": {
                    "color": "#cbd5e1",
                    "font-size": "15px",
                    "text-align": "left",
                    "margin": "4px 0",
                    "border-radius": "12px",
                    "padding": "6px 12px",
                    "transition": "all 0.3s ease"
                },
                "nav-link-selected": {
                    "background": "linear-gradient(135deg,#6366f1,#22d3ee)",
                    "color": "#f8fafc",
                    "font-weight": "700",
                    "font-size": "16px",
                    "box-shadow": "0 8px 20px rgba(56,189,248,0.6)"
                },
                "icon-selected": {"color": "#f8fafc"},
            }
        )

        st.markdown("---")

        if st.button("Logout", use_container_width=True):
            logout()

    return selected
# ========================================
# --- MAIN APP ROUTING ---
# ========================================

token = st.session_state.get("jwt_token")
payload = verify_token(token) if token else None

if payload:
    # Restore user context
    st.session_state["user"] = payload.get("sub")
    st.session_state["role"] = payload.get("role", "user")

    # 🔹 SIDEBAR CONTROLS PAGE
    selected = sidebar_navigation()

    if selected == "Chat":
        dashboard_page()
    elif selected == "Summarize":
        summarization_page()
    elif selected == "Knowledge Graph":
        knowledge_graph_page()
    elif selected == "Readability":
        readability_page()
    elif selected == "Admin":
        admin_dashboard()

else:
    # 🔓 PUBLIC ROUTES
    page = st.session_state.get("page", "login")

    if page == "signup":
        signup_page()
    elif page == "otp":
        otp_verify_page()
    elif page == "forgot":
        forgot_password_page()
    elif page == "forgot_verify":
        forgot_verify_page()
    elif page == "reset":
        reset_password_page()
    else:
        login_page()

Overwriting app.py


## Cell 9: Auto-Ingest PDFs → FAISS Index

In [ ]:
# Cell: Ingest PDFs from Google Drive into FAISS vector index
import os, vector_store

try:
    vector_store.get_embedder()
except:
    pass

docs_dir = os.path.join(os.environ.get('APP_DIR', '.'), 'documents')
print('🔍 Scanning Google Drive for PDFs...')
new_chunks = vector_store.ingest_documents(docs_dir)

if new_chunks > 0:
    print(f'✅ Ingested {new_chunks} new chunks into FAISS!')
else:
    print('⚡ FAISS index already up to date. Fast boot!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔍 Scanning Google Drive for PDFs...
Parsing new document: Central_PMFBY_Revamped%20Operational%20Guidelines_17th%20August%202020.pdf
Parsing new document: Central_JalJeevanMission_JJM_Operational_Guidelines.pdf
Parsing new document: Central_PMAY_Urban_Operational-Guidelines-of-PMAY-U.pdf
Parsing new document: Central_SmartCities_Resource_Doc_1723187622_Making_a_City_Smart_Learnings_from_the_Smart_Cities_Mission.pdf
Parsing new document: Central_PMJDY_PMJDY_Brochure_ENG.pdf
Parsing new document: Central_PMSBY_PMJJBY_Rules.pdf
Parsing new document: Central_NEP_2020_NEP_Final_English_0.pdf
Parsing new document: Central_PMKVY_4_PMKVY-4.0-Guidelines_final-copy.pdf
Parsing new document: Central_ScholarshipSC_ST_Guidelines-SCA%20to%20SCSP.pdf
Parsing new document: Central_DigitalIndia_Brochure.pdf
Parsing new document: Central_StartupIndia_198117.pdf
Parsing new document: Central_SocialJustice_AR_32691723633555.pdf
Parsing new document: Central_VCF_ScheduledCastes_56961654673488.pdf
Parsing n

Parsing new document: Central_PMGSY_prebidnotice.pdf
Parsing new document: Central_PMGSY_M3%20Exercise1%20Producing%20final%20map.pdf
Parsing new document: Central_PMGSY_Times%20of%20India_Newspapaer_12June%202024_0.pdf
Parsing new document: Central_PMGSY_AS_RDInternationalConferencePPTFinalVersion_24_5.pdf
Parsing new document: Central_PMGSY_3.%20Shri%20S%20Sai%20Baba-Sustainable%20design%20of%20long%20span%20bridges_0.pdf
Parsing new document: Central_PMGSY_WirtgenIndiaPresentation.pdf
Parsing new document: Central_PMGSY_FinalRFPdocument.pdf
Parsing new document: Central_PMGSY_35th%20Road%20Safety%20Week.pdf
Parsing new document: Central_PMGSY_ExperienceonuseofCoirinRuralRoads_DrSunithaV.pdf
Parsing new document: Central_PMGSY_L02-Construction%20and%20Quality%20control%20of%20Subgrade%20and%20%20Granular%20layers.pdf
Parsing new document: Central_PMGSY_format_NQM.pdf
Parsing new document: Central_PMGSY_Vision-Document-Booklet1.pdf
Parsing new document: Central_PMGSY_PPTModularBridges

Parsing new document: Central_Education_PIB2229093.pdf
Parsing new document: Central_Education_PIB2226698.pdf
Parsing new document: Central_Education_PIB2227059.pdf
Parsing new document: Central_Education_Advetisement%20for%20the%20Post%20of%20Member%20Secretary%20ICHR.pdf
Parsing new document: Central_Education_cvo_moe.pdf
Parsing new document: Central_Education_Advertisement_VC_Hyderabad.pdf
Parsing new document: Central_Education_dosel-Sept23-hi.pdf
Parsing new document: Central_Education_Aug23_dohe_hi.pdf
Parsing new document: Central_Education_dohe-mar25-hi.pdf
Parsing new document: Central_Education_Advertisement_Secretary_NCERT_0.pdf
Parsing new document: Central_NITI_CitizensCharter-of-NITI-Aayog290921.pdf
Parsing new document: Central_ESIC_Extension_of_last_date_of_receipt_of_applications_for_the_post_of_IMO_Gr_II_in_ESIC_through_PRATIBHA_Setu_portal_CMSE_2024_conducted_by_UPSC_English_1771587974.pdf
Parsing new document: Central_ESIC_Details_of_candidates_of_Combined_Medical_

## Cell 10: Load Secrets

In [ ]:
from google.colab import userdata
import os

SECRETS = {
    'JWT_SECRET_KEY':     userdata.get('JWT_SECRET_KEY'),
    'EMAIL_ID':           userdata.get('EMAIL_ID'),
    'EMAIL_APP_PASSWORD': userdata.get('EMAIL_APP_PASSWORD'),
    'ADMIN_EMAIL_ID':     userdata.get('ADMIN_EMAIL_ID'),
    'ADMIN_PASSWORD':     userdata.get('ADMIN_PASSWORD'),
    'NGROK_AUTHTOKEN':    userdata.get('NGROK_AUTHTOKEN'),
}
print('✅ Secrets loaded!')

✅ Secrets loaded!


## Cell 11: 🚀 Launch App

In [ ]:
from pyngrok import ngrok, conf
import subprocess, time, os, requests

env = os.environ.copy()
env['JWT_SECRET_KEY']     = SECRETS['JWT_SECRET_KEY']
env['EMAIL_ID']           = SECRETS['EMAIL_ID']
env['EMAIL_APP_PASSWORD'] = SECRETS['EMAIL_APP_PASSWORD']
env['ADMIN_EMAIL_ID']     = SECRETS['ADMIN_EMAIL_ID']
env['ADMIN_PASSWORD']     = SECRETS['ADMIN_PASSWORD']
env['APP_DIR']            = os.environ.get('APP_DIR', '.')

NGROK_TOKEN = SECRETS['NGROK_AUTHTOKEN']
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
time.sleep(2)

# Kill any existing streamlit
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

# Launch Streamlit
subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port=8501',
     '--server.headless=true',
     '--browser.gatherUsageStats=false'],
    env=env,
    stdout=open('streamlit.log', 'w'),
    stderr=subprocess.STDOUT
)
time.sleep(5)

public_url = ngrok.connect(8501)
print('🛡️ PolicyNav is LIVE!')
print('🌐 URL:', public_url)

🛡️ PolicyNav is LIVE!
🌐 URL: NgrokTunnel: "https://apochromatic-courtney-overplainly.ngrok-free.dev" -> "http://localhost:8501"
